# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from os.path import join as ospj
from glob import glob
import sys

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import ast
import seaborn as sns
from scipy import stats
from matplotlib.gridspec import GridSpec

mpl.style.use('figures.mplstyle')
MM_TO_IN = 1/25.4

from config import CONFIG
sys.path.append(CONFIG.tools_dir)

#stuff we use alot in this analysis
from iEEG_helper_functions import *
from scipy.stats import pearsonr
from statannotations.Annotator import Annotator
from heatmap_gen import PDFHeatmapGenerator
from scipy.stats import kruskal
from matplotlib.cm import ScalarMappable
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
from scipy.stats import mannwhitneyu
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests

import warnings
warnings.filterwarnings("ignore")


# Define color scheme and brain region categories
COLORS = {
    'teal': '#99D7CF',
    'yellow': '#E7CE95',
    'brown': '#8B4513',
    'gray': '#808080'
}

In [ ]:
from scipy.stats import mannwhitneyu
import itertools

def cliffs_delta(x, y):
    """Cliff's delta = (2*U - n1*n2) / (n1*n2), ranges from -1 to +1."""
    u_stat, _ = mannwhitneyu(x, y, alternative='two-sided')
    return (2 * u_stat - len(x) * len(y)) / (len(x) * len(y))

# Figure 1

## inter data

In [ ]:
analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_inter.csv'))
print(f'Loaded inter-stim data: {analysis_data.shape}')


In [ ]:
analysis_data.subject_id.unique()

In [ ]:
# analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'pre_post_optimized-inter_v2.csv'))

# analysis_data = analysis_data[analysis_data['subject_id'].isin(CONFIG.good_pts)]

# #what we did originally 
# analysis_data = analysis_data[
#     ~analysis_data['roi_rec_ch'].isin(['white-matter', 'outside-brain']) & 
#     analysis_data['roi_rec_ch'].notna()
# ].copy()

# baseline24 = pd.read_csv(ospj(CONFIG.dataset_dir, '24hr_merged_data.csv'))
# baseline24 = baseline24[['subject_id','recording_ch','mean_spike_rate_baseline']]
# baseline24 = baseline24[baseline24.subject_id.isin(CONFIG.good_pts)]
# baseline24 = baseline24.rename(columns = {'mean_spike_rate_baseline':'24hr_baseline', 'recording_ch':'recording_ch_name'})
# baseline24 = baseline24.groupby(['subject_id', 'recording_ch_name'])['24hr_baseline'].mean().reset_index()

# # # merge
# analysis_data = analysis_data.merge(baseline24, on = ['subject_id','recording_ch_name'], how = 'left')

# pt_distances = pd.read_csv(ospj(CONFIG.data_dir, 'localisations', 'all_locs_distances.csv'))
# near_mask = pt_distances['distance_mm'] >= 0
# near_distances = pt_distances[near_mask]
# near_distances = near_distances.rename(columns = {'label_1':'stim_ch_first',
#                                                  'label_2':'recording_ch_name',
#                                                  'roi_1':'roi_stim_ch',
#                                                  'roi_2':'roi_rec_ch'})
# # Create a new column for the first stim channel (before the "-")
# analysis_data['stim_ch_first'] = analysis_data['stim_ch'].str.split('-').str[0]
# # Merge using the new stim_ch_first column instead of stim_ch
# analysis_data = analysis_data.merge(
#     near_distances[['stim_ch_first', 'recording_ch_name', 'distance_mm','subject_id']],
#     on = ['subject_id','stim_ch_first', 'recording_ch_name'],
#     how='inner'
# )
# print(analysis_data.shape)

In [ ]:
#load SOZ information
# soz = pd.read_excel(ospj(CONFIG.data_dir, 'pt-metadata', 'Manual validation.xlsx'), sheet_name='SOZ', usecols = ['name','SOZ electrode','region','lateralization'])
# soz = soz[soz.name.str.contains('HUP')]
# soz['name'] = soz['name'].str.replace('HUP', '').astype(int)
# # Turn SOZ electrode to a list: split by comma if not null, else empty list
# soz['SOZ electrode'] = soz['SOZ electrode'].apply(
#     lambda x: [e.strip() for e in str(x).split(',')] if pd.notnull(x) and str(x).strip() != '' else []
# )
# soz = soz.rename(columns={'name':'subject_id'})
# soz.head(5)


In [ ]:
# #####################
# # BUILD EXPANDED SOZ SET PER PATIENT (SOZ + channels within 10mm)
# # Uses soz electrode list from metadata and pt_distances for proximity
# #####################

# # Build a lookup: subject_id -> set of original SOZ electrode labels
# soz_lookup = soz.set_index('subject_id')['SOZ electrode'].to_dict()

# # For each patient, expand SOZ to include any channel within 10mm of a SOZ electrode
# expanded_soz_lookup = {}

# for subject_id, soz_chs in soz_lookup.items():
#     soz_set = set(soz_chs)
#     if not soz_set:
#         expanded_soz_lookup[subject_id] = soz_set
#         continue

#     pt_dists = pt_distances[pt_distances['subject_id'] == subject_id]

#     # Find pairs where one side is a SOZ channel and distance <= 10mm
#     nearby_mask = (
#         (pt_dists['label_1'].isin(soz_set) | pt_dists['label_2'].isin(soz_set)) &
#         (pt_dists['distance_mm'] <= 10)
#     )
#     nearby_pairs = pt_dists[nearby_mask]

#     nearby_chs = set(nearby_pairs['label_1']) | set(nearby_pairs['label_2'])
#     expanded_soz_lookup[subject_id] = soz_set | nearby_chs

# # Create the two new columns on analysis_data
# analysis_data['soz_stim_ch_10mm'] = analysis_data.apply(
#     lambda row: row['stim_ch_first'] in expanded_soz_lookup.get(row['subject_id'], set()),
#     axis=1
# )

# analysis_data['soz_rec_ch_10mm'] = analysis_data.apply(
#     lambda row: row['recording_ch_name'] in expanded_soz_lookup.get(row['subject_id'], set()),
#     axis=1
# )

# print(f"soz_stim_ch_10mm True: {analysis_data['soz_stim_ch_10mm'].sum()}")
# print(f"soz_rec_ch_10mm  True: {analysis_data['soz_rec_ch_10mm'].sum()}")

# # Sanity check: compare original vs expanded
# print(f"\nOriginal soz_stim_ch True: {analysis_data['soz_stim_ch'].sum()}")
# print(f"Original soz_rec_ch  True: {analysis_data['soz_rec_ch'].sum()}")

In [ ]:
#####################
# GET HIGH SPIKING CHANNELS FROM 24HR BASELINE
#####################

# threshold = analysis_data["24hr_baseline"].quantile(0.86)
# For each subject_id, get a list of dicts with both recording_ch and recording_ch_name
threshold = 1
print(threshold)
high_baseline_chs = (
    analysis_data[analysis_data["24hr_baseline"] > threshold]
    .groupby("subject_id")[["recording_ch", "recording_ch_name"]]
    .apply(lambda df: df.drop_duplicates().to_dict('records'))
    .to_dict()
)
print(len(high_baseline_chs.keys()))

## Change over distance

In [ ]:
# Create a set of (subject_id, recording_ch) tuples from high_baseline_chs for efficient lookup
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))

# Filter for high baseline channels (IZ)
which = analysis_data[
    analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1)
].copy()

# Create distance bins
n_bins = 30
distance_bins = pd.cut(which['distance_mm'], bins=n_bins)

# Add the distance bins to the dataframe for easier grouping
which_with_bins = which.copy()
which_with_bins['distance_bin'] = distance_bins

# Calculate patient-specific statistics for each bin
patient_bin_stats = which_with_bins.groupby(['subject_id', 'distance_bin'])['change_during_to_inter'].agg(['mean', 'std', 'count']).reset_index()
patient_bin_stats['sem'] = patient_bin_stats['std'] / np.sqrt(patient_bin_stats['count'])
patient_bin_stats['bin_center'] = patient_bin_stats['distance_bin'].apply(lambda x: x.mid)

# Calculate overall across-patient statistics for each bin
# First get patient means for each bin, then average across patients
patient_means_per_bin = patient_bin_stats.groupby('distance_bin')['mean'].agg(['mean', 'std', 'count']).reset_index()
patient_means_per_bin['sem'] = patient_means_per_bin['std'] / np.sqrt(patient_means_per_bin['count'])
patient_means_per_bin['bin_center'] = patient_means_per_bin['distance_bin'].apply(lambda x: x.mid)

fig, ax = plt.subplots(figsize=(80*MM_TO_IN, 55*MM_TO_IN))

# Plot individual patient lines in light blue (each line = one patient's binned statistics)
unique_patients = patient_bin_stats['subject_id'].unique()
first_patient = True
for patient_id in unique_patients:
    patient_data = patient_bin_stats[patient_bin_stats['subject_id'] == patient_id]
    # Only plot if patient has data in multiple bins
    if len(patient_data) > 1:
        label = 'Individual patient mean (SEM)' if first_patient else '_nolegend_'
        ax.errorbar(patient_data['bin_center'], patient_data['mean'], 
                   yerr=patient_data['sem'], fmt='o-', capsize=1, capthick=0.1, 
                   markersize=1, color='#99D7CF', alpha=0.5, linewidth=0.5, label=label)
        first_patient = False

# Plot across-patient average in black (average of patient means per bin)
ax.errorbar(patient_means_per_bin['bin_center'], patient_means_per_bin['mean'], 
           yerr=patient_means_per_bin['sem'], fmt='o-', capsize=2, capthick=0.7, 
           markersize=1, color='black', linewidth=0.7, label='Across-patient mean (SEM)',zorder=10)


ax.axhline(y=0, color='red', linestyle='--', alpha=0.8, zorder = 9)
ax.set_xlabel('Distance (mm) of stimulation from IZ channels')
ax.set_ylabel('Mean Δ Spike Rate ± SEM \n (spikes/min)')
# ax.set_title('Δ Spike Rate by increasing stimulation distance (Rec: IZ)')
# ax.set_yscale('symlog', linthresh=0.1, linscale=1)

ax.set_ylim(-10, 20)
ax.set_xlim(0, 80)

ax.legend()
plt.tight_layout()

## Main plot - during/inter

In [ ]:
import matplotlib
# Create a set of (subject_id, recording_ch) tuples from high_baseline_chs for efficient lookup
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))

# Filter for nearby stimulation (<=40mm) AND high baseline channels
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40) & 
    (analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1))
].copy()
print(f"Total observations with distance ≤ 40mm and high baseline: {len(nearby_data)}")

# Calculate average spike rates per patient for during and inter
patient_avg_nearby = nearby_data.groupby('subject_id').agg({
    'spike_rate_during': 'mean',
    'spike_rate_inter': 'mean'
}).reset_index()

# Only keep patients with both measures (non-NaN)
valid_patients_nearby = patient_avg_nearby.dropna(subset=['spike_rate_during', 'spike_rate_inter'])
print(f"Number of patients with both during and inter measures (≤40mm): {len(valid_patients_nearby)}")

x_labels = ['Inter-Stimulation', 'During Stimulation']
x = [0, 1]

fig, ax = plt.subplots(1, 1, figsize=(80 * MM_TO_IN, 60 * MM_TO_IN))

# Plot paired lines for each patient using raw spike rates
for i, row in valid_patients_nearby.iterrows():
    y = [row['spike_rate_inter'], row['spike_rate_during']]
    color_line = COLORS['teal'] if row['spike_rate_during'] > row['spike_rate_inter'] else COLORS['yellow']
    ax.plot([x[0], x[1]], [y[0], y[1]], color=color_line, alpha=0.5, linewidth=1)

# Plot medians
means = [
    valid_patients_nearby['spike_rate_inter'].median(),
    valid_patients_nearby['spike_rate_during'].median()
]
ax.plot(x, means, 'k-', linewidth=2, label='Median')

# Scatter points
ax.scatter([x[0]]*len(valid_patients_nearby), valid_patients_nearby['spike_rate_inter'],
           c=COLORS['yellow'], label='Inter-Stimulation', s=50, zorder=10, edgecolor='black', linewidth=0.1)
ax.scatter([x[1]]*len(valid_patients_nearby), valid_patients_nearby['spike_rate_during'],
           c=COLORS['teal'], label='During Stimulation', s=50, zorder=10, edgecolor='black', linewidth=0.1)

# Apply symlog scale so axis reads 0, 1, 10, 100, etc.
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize = 6)
ax.set_ylabel('Spike Rate (spikes/min; log scale)')
# ax.set_title('Overall Δ Spike Rate (Rec: IZ, Stim: ≤40mm)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Build paired DataFrame for statsannotations
paired_data = []
for i, row in valid_patients_nearby.iterrows():
    paired_data.extend([
        {'subject_id': row['subject_id'], 'Timepoint': 'Inter-Stimulation', 'Spike Rate': row['spike_rate_inter']},
        {'subject_id': row['subject_id'], 'Timepoint': 'During Stimulation', 'Spike Rate': row['spike_rate_during']}
    ])
paired_df = pd.DataFrame(paired_data)

pairs = [('Inter-Stimulation', 'During Stimulation')]
annotator = Annotator(ax, pairs, data=paired_df, x="Timepoint", y="Spike Rate", order=x_labels)
annotator.configure(
    test='Wilcoxon',
    text_format='star',
    loc='inside',
    hide_non_significant=False,
    verbose=True
)
annotator.apply_and_annotate()
ax.set_xlim(-0.2, 1.2)

plt.tight_layout()
# plt.savefig(ospj(CONFIG.fig_dir, 'symlog_during_inter_spike_rates_nearby.pdf'), bbox_inches='tight', dpi=300)
plt.show()

# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
# During Stimulation
q25_during = valid_patients_nearby['spike_rate_during'].quantile(0.25)
q75_during = valid_patients_nearby['spike_rate_during'].quantile(0.75)
median_during = valid_patients_nearby['spike_rate_during'].median()
print(f"During Stimulation - Median (25% - 75%): {median_during:.2f} ({q25_during:.2f} - {q75_during:.2f})")

# Inter-Stimulation
q25_inter = valid_patients_nearby['spike_rate_inter'].quantile(0.25)
q75_inter = valid_patients_nearby['spike_rate_inter'].quantile(0.75)
median_inter = valid_patients_nearby['spike_rate_inter'].median()
print(f"Inter-Stimulation  - Median (25% - 75%): {median_inter:.2f} ({q25_inter:.2f} - {q75_inter:.2f})")

In [ ]:
# Cliff's Delta for each pairwise comparison (raw spike rates)
inter    = valid_patients_nearby['spike_rate_inter'].values
during = valid_patients_nearby['spike_rate_during'].values

def interpret_cliffs(d):
    return 'negligible' if abs(d) < 0.147 else 'small' if abs(d) < 0.33 else 'medium' if abs(d) < 0.474 else 'large'

d_inter_during = cliffs_delta(during, inter)

print("=== CLIFF'S DELTA ===")
print(f"During vs Inter:   {d_inter_during:.3f} ({interpret_cliffs(d_inter_during)})")

## Main plot - pre/post/during

In [ ]:
analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_prepost.csv'))
print(f'Loaded pre-post data: {analysis_data.shape}')

# GET HIGH SPIKING CHANNELS FROM 24HR BASELINE
#####################
threshold = analysis_data["24hr_baseline"].quantile(0.90)
# For each subject_id, get a list of dicts with both recording_ch and recording_ch_name
threshold = 1
print(threshold)
high_baseline_chs = (
    analysis_data[analysis_data["24hr_baseline"] > threshold]
    .groupby("subject_id")[["recording_ch", "recording_ch_name"]]
    .apply(lambda df: df.drop_duplicates().to_dict('records'))
    .to_dict()
)
print(len(high_baseline_chs.keys()))


In [ ]:
import matplotlib

# Create a set of (subject_id, recording_ch) tuples from high_baseline_chs for efficient lookup
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))

# Filter for nearby stimulation (<=40mm) AND high baseline channels
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40) & 
    (analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1))
].copy()
print(f"Total observations with distance ≤ 40mm and high baseline: {len(nearby_data)}")

# Calculate average spike rates per patient for pre, during, and post
patient_avg_nearby = nearby_data.groupby('subject_id').agg({
    'spike_rate_pre': 'mean',
    'spike_rate_during': 'mean',
    'spike_rate_post': 'mean'
}).reset_index()

# Only keep patients with all three measures (non-NaN)
valid_patients_nearby = patient_avg_nearby.dropna(subset=['spike_rate_pre', 'spike_rate_during', 'spike_rate_post'])
print(f"Number of patients with pre, during, and post measures (≤40mm): {len(valid_patients_nearby)}")

x_labels = ['Pre-Stimulation', 'During Stimulation', 'Post-Stimulation']
x = [0, 1, 2]

fig, ax = plt.subplots(1, 1, figsize=(100 * MM_TO_IN, 60 * MM_TO_IN))

# Plot paired lines for each patient using raw spike rates
for i, row in valid_patients_nearby.iterrows():
    y = [row['spike_rate_pre'], row['spike_rate_during'], row['spike_rate_post']]
    # Pre -> During segment
    color_pre_dur = COLORS['teal'] if row['spike_rate_during'] > row['spike_rate_pre'] else COLORS['yellow']
    ax.plot([x[0], x[1]], [y[0], y[1]], color=color_pre_dur, alpha=0.5, linewidth=1)
    # During -> Post segment
    color_dur_post = COLORS['brown'] if row['spike_rate_post'] > row['spike_rate_during'] else COLORS['teal']
    ax.plot([x[1], x[2]], [y[1], y[2]], color=color_dur_post, alpha=0.5, linewidth=1)

# Plot medians across all three timepoints
medians = [
    valid_patients_nearby['spike_rate_pre'].median(),
    valid_patients_nearby['spike_rate_during'].median(),
    valid_patients_nearby['spike_rate_post'].median()
]
ax.plot(x, medians, 'k-', linewidth=2, label='Median')

# Scatter points
ax.scatter([x[0]]*len(valid_patients_nearby), valid_patients_nearby['spike_rate_pre'],
           c=COLORS['yellow'], label='Pre-Stimulation', s=50, zorder=10, edgecolor='black', linewidth=0.1)
ax.scatter([x[1]]*len(valid_patients_nearby), valid_patients_nearby['spike_rate_during'],
           c=COLORS['teal'], label='During Stimulation', s=50, zorder=10, edgecolor='black', linewidth=0.1)
ax.scatter([x[2]]*len(valid_patients_nearby), valid_patients_nearby['spike_rate_post'],
           c=COLORS['brown'], label='Post-Stimulation', s=50, zorder=10, edgecolor='black', linewidth=0.1)

# Apply symlog scale so axis reads 0, 1, 10, 100, etc.
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize = 6)
ax.set_ylabel('Spike Rate (spikes/min; log scale)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Build paired DataFrame for statsannotations
paired_data = []
for i, row in valid_patients_nearby.iterrows():
    paired_data.extend([
        {'subject_id': row['subject_id'], 'Timepoint': 'Pre-Stimulation', 'Spike Rate': row['spike_rate_pre']},
        {'subject_id': row['subject_id'], 'Timepoint': 'During Stimulation', 'Spike Rate': row['spike_rate_during']},
        {'subject_id': row['subject_id'], 'Timepoint': 'Post-Stimulation', 'Spike Rate': row['spike_rate_post']}
    ])
paired_df = pd.DataFrame(paired_data)

pairs = [
    ('Pre-Stimulation', 'During Stimulation'),
    ('During Stimulation', 'Post-Stimulation'),
    ('Pre-Stimulation', 'Post-Stimulation')
]
annotator = Annotator(ax, pairs, data=paired_df, x="Timepoint", y="Spike Rate", order=x_labels)
annotator.configure(
    test='Wilcoxon',
    text_format='star',
    loc='inside',
    comparisons_correction='bonf',
    hide_non_significant=False,
    verbose=True
)
annotator.apply_and_annotate()
ax.set_xlim(-0.2, 2.2)

plt.tight_layout()
# plt.savefig(ospj(CONFIG.fig_dir, 'symlog_pre_during_post_spike_rates_nearby.pdf'), bbox_inches='tight', dpi=300)
plt.show()

# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
for label, col in [
    ("Pre-Stimulation", 'spike_rate_pre'),
    ("During Stimulation", 'spike_rate_during'),
    ("Post-Stimulation", 'spike_rate_post'),
]:
    arr = valid_patients_nearby[col]
    median = arr.median()
    q25 = arr.quantile(0.25)
    q75 = arr.quantile(0.75)
    print(f"{label:20} - Median: {median:.2f} ({q25:.2f} - {q75:.2f})")

In [ ]:
# Cliff's Delta for each pairwise comparison (raw spike rates)
pre    = valid_patients_nearby['spike_rate_pre'].values
during = valid_patients_nearby['spike_rate_during'].values
post   = valid_patients_nearby['spike_rate_post'].values

def interpret_cliffs(d):
    return 'negligible' if abs(d) < 0.147 else 'small' if abs(d) < 0.33 else 'medium' if abs(d) < 0.474 else 'large'

d_pre_during = cliffs_delta(during, pre)
d_during_post = cliffs_delta(post, during)
d_pre_post = cliffs_delta(post, pre)

print("=== CLIFF'S DELTA ===")
print(f"During vs Pre:   {d_pre_during:.3f} ({interpret_cliffs(d_pre_during)})")
print(f"Post vs During:  {d_during_post:.3f} ({interpret_cliffs(d_during_post)})")
print(f"Post vs Pre:     {d_pre_post:.3f} ({interpret_cliffs(d_pre_post)})")

## How long does rebound last?

In [ ]:
rebound_dir = ospj(CONFIG.results_dir, 'spike_detection_derivatives', 'rebound_spike_detection_try2')
rebound_output = glob(ospj(rebound_dir, '*_output.csv'))
rebound_channels = glob(ospj(rebound_dir, '*_channels.csv'))

In [ ]:
# (hardcoded path removed)
# )
# (hardcoded path removed)
# )

In [ ]:
# display(rebound_output)
# display(channels_example[channels_example['window_idx'] == 5])

In [ ]:
# ── config ────────────────────────────────────────────────────────────────────
WINDOW_DURATION_S = 50
BIN_SIZE_S        = 10
N_BINS            = WINDOW_DURATION_S // BIN_SIZE_S  # 5 bins per window

# Build fs lookup from rebound dataset (fs column is already merged in)
fs_lookup = (
    pd.read_csv(ospj(CONFIG.dataset_dir, "analysis_data_rebound.csv"),
                usecols=["subject_id", "fs"])
    .drop_duplicates("subject_id")
    .set_index("subject_id")["fs"]
    .to_dict()
)


In [ ]:
# (hardcoded path removed)


In [ ]:
rebound_spike_rates = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_rebound.csv'))
print(f'Loaded rebound data: {rebound_spike_rates.shape}')


In [ ]:
rebound_spike_rates

### reboiund spike rate, just spikes/min 

In [ ]:
stim_times = pd.read_csv(ospj(CONFIG.dataset_dir, 'stim_channels.csv'))
print(f'Loaded stim channels: {stim_times.shape}')


In [ ]:
stim_times = pd.read_csv(ospj(CONFIG.dataset_dir, 'stim_channels.csv'))
print(f'Loaded stim channels: {stim_times.shape}')


In [ ]:
# ── plot: rebound spike rate — individual patients + across-patient mean ± SEM ─

TYPE_COLORS = {
    'standard_last_stim': COLORS['yellow'],
    'dense_post':         COLORS['teal'],
    'sparse_post':        COLORS['gray'],
}

# step 1: per-patient mean ± SEM across channels at each bin_start_s
pt_stats = (
    rebound_filtered
    .groupby(['subject_id', 'bin_start_s', 'window_type'])['spikes_per_min']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
pt_stats['sem'] = pt_stats['std'] / np.sqrt(pt_stats['count'])

# step 2: across-patient mean ± SEM (average of per-patient means)
pop_stats = (
    pt_stats
    .groupby(['bin_start_s', 'window_type'])['mean']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('bin_start_s')
)
pop_stats['sem'] = pop_stats['std'] / np.sqrt(pop_stats['count'])

fig, ax = plt.subplots(figsize=(160 * MM_TO_IN, 60 * MM_TO_IN))

# shade background by window type
for wtype, grp in pop_stats.groupby('window_type'):
    color = TYPE_COLORS.get(wtype, COLORS['gray'])
    for bs in grp['bin_start_s']:
        ax.axvspan(bs, bs + BIN_SIZE_S, alpha=0.15, color=color, linewidth=0)

# individual patient lines
patients = pt_stats['subject_id'].unique()
first_pt = True
for hup in patients:
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient mean (SEM)' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2,
                pt['mean'],
                yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=COLORS['teal'],
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False

# across-patient mean ± SEM
x      = pop_stats['bin_start_s'] + BIN_SIZE_S / 2
mean_v = pop_stats['mean']
sem_v  = pop_stats['sem']

ax.errorbar(x, mean_v,
            yerr=sem_v,
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)

ax.axvline(0, color='red', linewidth=0.8, linestyle='--', label='Last stim (t=0)', zorder=9)

ax.set_xlabel('Time from last stim (s)', fontsize=7)
ax.set_ylabel('Spikes/min', fontsize=7)
ax.tick_params(labelsize=6)

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, alpha=0.4, label=t) for t, c in TYPE_COLORS.items()]
legend_handles += [
    plt.Line2D([0], [0], color=COLORS['teal'], alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient mean (SEM)'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
    plt.Line2D([0], [0], color='red', linestyle='--',
               linewidth=0.8, label='Last stim (t=0)'),
]
ax.legend(handles=legend_handles, fontsize=5, frameon=False)

n_pts = rebound_filtered['subject_id'].nunique()
ax.set_title(
    f'Rebound spike rate — 10s bins\n'
    f'Good pts · IZ channels (top 90th pct) · ≤40mm from stim  (n={n_pts})',
    fontsize=8
)
# ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# ── normalize rebound to pre-stim spike rate from analysis_data ──────────────
# uses the same 'spike_rate_pre' that drives the pre/during/post plots

pre_baseline = (
    nearby_data.groupby('subject_id')['spike_rate_pre']
    .mean()
    .rename('baseline_rate')
    .reset_index()
)

rebound_norm = rebound_filtered.merge(pre_baseline, on='subject_id', how='inner')
rebound_norm = rebound_norm[rebound_norm['baseline_rate'] > 0].copy()
rebound_norm['spikes_norm'] = rebound_norm['spikes_per_min'] / rebound_norm['baseline_rate']

# step 1: per-patient mean ± SEM across channels at each bin
pt_stats = (
    rebound_norm
    .groupby(['subject_id', 'bin_start_s', 'window_type'])['spikes_norm']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
pt_stats['sem'] = pt_stats['std'] / np.sqrt(pt_stats['count'])

# step 2: across-patient mean ± SEM
pop_stats = (
    pt_stats
    .groupby(['bin_start_s', 'window_type'])['mean']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('bin_start_s')
)
pop_stats['sem'] = pop_stats['std'] / np.sqrt(pop_stats['count'])

TYPE_COLORS = {
    'standard_last_stim': COLORS['yellow'],
    'dense_post':         COLORS['teal'],
    'sparse_post':        COLORS['gray'],
}

fig, ax = plt.subplots(figsize=(160 * MM_TO_IN, 60 * MM_TO_IN))

for wtype, grp in pop_stats.groupby('window_type'):
    color = TYPE_COLORS.get(wtype, COLORS['gray'])
    for bs in grp['bin_start_s']:
        ax.axvspan(bs, bs + BIN_SIZE_S, alpha=0.15, color=color, linewidth=0)

first_pt = True
for hup in pt_stats['subject_id'].unique():
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2,
                pt['mean'], yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=COLORS['teal'],
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False

x = pop_stats['bin_start_s'] + BIN_SIZE_S / 2
ax.errorbar(x, pop_stats['mean'], yerr=pop_stats['sem'],
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)

ax.axhline(1, color='red', linewidth=0.8, linestyle='--', label='Pre-stim baseline (=1)', zorder=9)
ax.axvline(0, color='gray', linewidth=0.6, linestyle=':', zorder=9)

ax.set_xlabel('Time from last stim (s)', fontsize=7)
ax.set_ylabel('Spike rate\n(normalized to pre-stim)', fontsize=7)
ax.tick_params(labelsize=6)

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, alpha=0.4, label=t) for t, c in TYPE_COLORS.items()]
legend_handles += [
    plt.Line2D([0], [0], color=COLORS['teal'], alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
    plt.Line2D([0], [0], color='red', linestyle='--',
               linewidth=0.8, label='Pre-stim baseline (=1)'),
]
ax.legend(handles=legend_handles, fontsize=5, frameon=False)

n_pts = rebound_norm['subject_id'].nunique()
ax.set_title(
    f'Rebound spike rate normalized to pre-stim (spike_rate_pre)\n'
    f'Good pts · IZ channels · ≤40mm from stim  (n={n_pts})',
    fontsize=8
)
plt.tight_layout()
ax.set_ylim(0, 200)
plt.show()

### working figure

#### working

In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin['stim_sz_mtl'] = stimsz_erin['Stim seizure elicited in MTL'].map({'Yes': 1, 'No': 0})
stimsz_erin = stimsz_erin[['subject_id', 'stim_sz_mtl']].dropna(subset=['stim_sz_mtl'])
stimsz_erin['stim_sz_mtl'] = stimsz_erin['stim_sz_mtl'].astype(int)
no_stim_sz_pts = stimsz_erin[stimsz_erin['stim_sz_mtl'] == 0]['subject_id']


In [ ]:
np.array(no_stim_sz_pts)

In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin['stim_sz_mtl'] = stimsz_erin['Stim seizure elicited in MTL'].map({'Yes': 1, 'No': 0})
stimsz_erin = stimsz_erin[['subject_id', 'stim_sz_mtl']].dropna(subset=['stim_sz_mtl'])
stimsz_erin['stim_sz_mtl'] = stimsz_erin['stim_sz_mtl'].astype(int)
no_stim_sz_pts = stimsz_erin[stimsz_erin['stim_sz_mtl'] == 0]['subject_id']
rebound_filtered_nostimsz = rebound_filtered[
    rebound_filtered['subject_id'].isin(no_stim_sz_pts)
].copy()
print(f"Patients before stim_sz exclusion: {rebound_filtered['subject_id'].nunique()}")
print(f"Patients after  stim_sz exclusion: {rebound_filtered_nostimsz['subject_id'].nunique()}")
# ── normalize to pre-stim spike rate from analysis_data ──────────────────────
pre_baseline = (
    nearby_data.groupby('subject_id')['spike_rate_pre']
    .mean()
    .rename('baseline_rate')
    .reset_index()
)
rebound_norm = rebound_filtered_nostimsz.merge(pre_baseline, on='subject_id', how='inner')
rebound_norm = rebound_norm[rebound_norm['baseline_rate'] > 0].copy()
rebound_norm['spikes_norm'] = rebound_norm['spikes_per_min'] / rebound_norm['baseline_rate']
# rebound_norm['spikes_norm'] = rebound_norm['spikes_per_min']
# step 1: per-patient mean ± SEM across channels at each bin
pt_stats = (
    rebound_norm
    .groupby(['subject_id', 'bin_start_s', 'window_type'])['spikes_norm']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
pt_stats['sem'] = pt_stats['std'] / np.sqrt(pt_stats['count'])
# step 2: across-patient mean ± SEM
pop_stats = (
    pt_stats
    .groupby(['bin_start_s', 'window_type'])['mean']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('bin_start_s')
)
pop_stats['sem'] = pop_stats['std'] / np.sqrt(pop_stats['count'])
PT_COLOR = '#4472C4'  # blue — distinct from all background regions
fig, ax = plt.subplots(figsize=(140 * MM_TO_IN, 51 * MM_TO_IN))
# hardcoded background bands by time range
ax.axvspan(-10,   0, alpha=0.20, color=COLORS['yellow'], linewidth=0)  # pre-stim
ax.axvspan(  0,  30, alpha=0.20, color=COLORS['teal'],   linewidth=0)  # dense post
ax.axvspan( 30, 280, alpha=0.2, color=COLORS['brown'],  linewidth=0)  # sparse post
# ax.axvspan( 40, 280, alpha=0.2, color=COLORS['gray'],  linewidth=0)  # sparse post
# individual patient lines
first_pt = True
for hup in pt_stats['subject_id'].unique():
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2,
                pt['mean'], yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=PT_COLOR,
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False
# across-patient mean ± SEM
x = pop_stats['bin_start_s'] + BIN_SIZE_S / 2
ax.errorbar(x, pop_stats['mean'], yerr=pop_stats['sem'],
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)
# ax.axhline(1, color='red', linewidth=0.8, linestyle='--', label='Pre-stim baseline (=1)', zorder=9)
ax.axvline(0, color='gray', linewidth=0.6, linestyle=':', zorder=9)
ax.set_xlabel('Time from last stim (s)')
ax.set_ylabel('Spike rate (spikes/min; log scale) \nnorm. to trial average pre-stim')
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))
ax.set_ylim(0, 100)
# ax.set_yscale('symlog', linthresh=0.1, linscale=0.5)
# ax.set_ylim(0, 100)  # cut off the negative region entirely
# ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
# ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 0.1, 1, 10, 100]))
from matplotlib.patches import Patch
legend_handles = [
    # Patch(color=COLORS['yellow'], alpha=1, label='Pre-stimulation (-10 \u2013 0s)'),
    # Patch(color=COLORS['teal'],   alpha=1, label='During Stimulation (0 \u2013 30s)'),
    # Patch(color=COLORS['brown'],  alpha=1, label='Post-stimulation (30s +)'),
    plt.Line2D([0], [0], color=PT_COLOR, alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient (SEM)'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
    # plt.Line2D([0], [0], color='red', linestyle='--',
    #            linewidth=0.8, label='Pre-stim baseline (=1)'),
]
ax.legend(handles=legend_handles, frameon=True, framealpha=0.5, edgecolor='black')
n_pts = rebound_norm['subject_id'].nunique()
print(f"n = {n_pts} patients in final plot")
ax.set_xlim(-10, 200)
# plt.savefig(ospj(CONFIG.fig_dir, 'rebound_spike_rate_normalized.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
TIME_SHIFT = -30  # shift so stim end = 0

fig, ax = plt.subplots(figsize=(140 * MM_TO_IN, 51 * MM_TO_IN))

# shifted background bands
ax.axvspan(-40, -30, alpha=0.20, color=COLORS['yellow'], linewidth=0)  # pre-stim
ax.axvspan(-30,   0, alpha=0.20, color=COLORS['teal'],   linewidth=0)  # during
ax.axvspan(  0, 250, alpha=0.20, color=COLORS['brown'],  linewidth=0)  # post

# individual patient lines
first_pt = True
for hup in pt_stats['subject_id'].unique():
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2 + TIME_SHIFT,
                pt['mean'], yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=PT_COLOR,
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False

# across-patient mean ± SEM
x = pop_stats['bin_start_s'] + BIN_SIZE_S / 2 + TIME_SHIFT
ax.errorbar(x, pop_stats['mean'], yerr=pop_stats['sem'],
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)

# stim onset/offset markers
ax.axvline(-30, color='gray', linewidth=0.6, linestyle=':', zorder=9)  # stim on
ax.axvline(  0, color='gray', linewidth=0.6, linestyle=':', zorder=9)  # stim off

ax.set_xlabel('Time from end of stim (s)')
ax.set_ylabel('Spike rate (spikes/min; log scale) \nnorm. to trial average pre-stim')

ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))
ax.set_ylim(0, 100)

legend_handles = [
    plt.Line2D([0], [0], color=PT_COLOR, alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient (SEM)'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
]
ax.legend(handles=legend_handles, frameon=True, framealpha=0.5, edgecolor='black')

ax.set_xlim(-40, 170)
ax.set_xticks(np.arange(-40, 171, 20))

n_pts = rebound_norm['subject_id'].nunique()
print(f"n = {n_pts} patients in final plot")
plt.show()

#### try inner trial pre-stim norm

In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin['stim_sz_mtl'] = stimsz_erin['Stim seizure elicited in MTL'].map({'Yes': 1, 'No': 0})
stimsz_erin = stimsz_erin[['subject_id', 'stim_sz_mtl']].dropna(subset=['stim_sz_mtl'])
stimsz_erin['stim_sz_mtl'] = stimsz_erin['stim_sz_mtl'].astype(int)
no_stim_sz_pts = stimsz_erin[stimsz_erin['stim_sz_mtl'] == 0]['subject_id']
rebound_filtered_nostimsz = rebound_filtered[
    rebound_filtered['subject_id'].isin(no_stim_sz_pts)
].copy()
print(f"Patients before stim_sz exclusion: {rebound_filtered['subject_id'].nunique()}")
print(f"Patients after  stim_sz exclusion: {rebound_filtered_nostimsz['subject_id'].nunique()}")
# ── normalize to pre-stim bin from the SAME clips being plotted ──────────────
# pre-stim window in this dataset is bin_start_s in [-10, 0)
pre_mask = (rebound_filtered_nostimsz['bin_start_s'] >= -10) & \
           (rebound_filtered_nostimsz['bin_start_s'] < 0)
pre_baseline = (
    rebound_filtered_nostimsz[pre_mask]
    .groupby('subject_id')['spikes_per_min']
    .mean()
    .rename('baseline_rate')
    .reset_index()
)
rebound_norm = rebound_filtered_nostimsz.merge(pre_baseline, on='subject_id', how='inner')
rebound_norm = rebound_norm[rebound_norm['baseline_rate'] > 0].copy()
rebound_norm['spikes_norm'] = rebound_norm['spikes_per_min'] / rebound_norm['baseline_rate']
# step 1: per-patient mean ± SEM across channels at each bin
pt_stats = (
    rebound_norm
    .groupby(['subject_id', 'bin_start_s', 'window_type'])['spikes_norm']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
pt_stats['sem'] = pt_stats['std'] / np.sqrt(pt_stats['count'])
# step 2: across-patient mean ± SEM
pop_stats = (
    pt_stats
    .groupby(['bin_start_s', 'window_type'])['mean']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('bin_start_s')
)
pop_stats['sem'] = pop_stats['std'] / np.sqrt(pop_stats['count'])
PT_COLOR = '#4472C4'
fig, ax = plt.subplots(figsize=(150 * MM_TO_IN, 51 * MM_TO_IN))
ax.axvspan(-10,   0, alpha=0.20, color=COLORS['yellow'], linewidth=0)
ax.axvspan(  0,  30, alpha=0.20, color=COLORS['teal'],   linewidth=0)
ax.axvspan( 30, 280, alpha=0.20, color=COLORS['brown'],  linewidth=0)
first_pt = True
for hup in pt_stats['subject_id'].unique():
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2,
                pt['mean'], yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=PT_COLOR,
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False
x = pop_stats['bin_start_s'] + BIN_SIZE_S / 2
ax.errorbar(x, pop_stats['mean'], yerr=pop_stats['sem'],
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)
ax.axhline(1, color='red', linewidth=0.8, linestyle='--', label='Pre-stim baseline (=1)', zorder=9)
ax.axvline(0, color='gray', linewidth=0.6, linestyle=':', zorder=9)
ax.set_xlabel('Time from last stim (s)')
ax.set_ylabel('Spike rate\n(norm. to pre-stim)')
from matplotlib.patches import Patch
legend_handles = [
    Patch(color=COLORS['yellow'], alpha=1, label='Pre-stimulation (-10 \u2013 0s)'),
    Patch(color=COLORS['teal'],   alpha=1, label='During Stimulation (0 \u2013 30s)'),
    Patch(color=COLORS['brown'],  alpha=1, label='Post-stimulation (30s +)'),
    plt.Line2D([0], [0], color=PT_COLOR, alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient (SEM)'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
    plt.Line2D([0], [0], color='red', linestyle='--',
               linewidth=0.8, label='Pre-stim baseline (=1)'),
]
ax.legend(handles=legend_handles, frameon=True, framealpha=0.5, edgecolor='black')
n_pts = rebound_norm['subject_id'].nunique()
print(f"n = {n_pts} patients in final plot")
ax.set_xlim(-10, 200)
plt.show()


In [ ]:
TIME_SHIFT = -30  # shift so stim end = 0

fig, ax = plt.subplots(figsize=(140 * MM_TO_IN, 51 * MM_TO_IN))

# shifted background bands
ax.axvspan(-40, -30, alpha=0.20, color=COLORS['yellow'], linewidth=0)  # pre-stim
ax.axvspan(-30,   0, alpha=0.20, color=COLORS['teal'],   linewidth=0)  # during
ax.axvspan(  0, 250, alpha=0.20, color=COLORS['brown'],  linewidth=0)  # post

# individual patient lines
first_pt = True
for hup in pt_stats['subject_id'].unique():
    pt = pt_stats[pt_stats['subject_id'] == hup].sort_values('bin_start_s')
    if len(pt) < 2:
        continue
    label = 'Individual patient' if first_pt else '_nolegend_'
    ax.errorbar(pt['bin_start_s'] + BIN_SIZE_S / 2 + TIME_SHIFT,
                pt['mean'], yerr=pt['sem'],
                fmt='o-', capsize=1, capthick=0.1,
                markersize=1, color=PT_COLOR,
                alpha=0.5, linewidth=0.5, label=label)
    first_pt = False

# across-patient mean ± SEM
x = pop_stats['bin_start_s'] + BIN_SIZE_S / 2 + TIME_SHIFT
ax.errorbar(x, pop_stats['mean'], yerr=pop_stats['sem'],
            fmt='o-', capsize=2, capthick=0.7,
            markersize=2, color='black', linewidth=0.9,
            label='Across-patient mean (SEM)', zorder=10)

# stim onset/offset markers
ax.axvline(-30, color='gray', linewidth=0.6, linestyle=':', zorder=9)  # stim on
ax.axvline(  0, color='gray', linewidth=0.6, linestyle=':', zorder=9)  # stim off

ax.set_xlabel('Time from end of stim (s)')
ax.set_ylabel('Spike rate (spikes/min; log scale) \nnorm. to trial average pre-stim')

ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))
ax.set_ylim(0, 100)

legend_handles = [
    plt.Line2D([0], [0], color=PT_COLOR, alpha=0.5, linewidth=0.5,
               marker='o', markersize=1, label='Individual patient (SEM)'),
    plt.Line2D([0], [0], color='black', linewidth=0.9,
               marker='o', markersize=2, label='Across-patient mean (SEM)'),
]
ax.legend(handles=legend_handles, frameon=True, framealpha=0.5, edgecolor='black')

ax.set_xlim(-40, 170)
ax.set_xticks(np.arange(-40, 171, 20))

n_pts = rebound_norm['subject_id'].nunique()
print(f"n = {n_pts} patients in final plot")
plt.show()

#### statistical test

In [ ]:
# ── 4-timepoint comparison: Pre, During (from nearby plot) vs 0–60s, 60–120s post (from rebound) ──
# Patients must appear in both datasets: valid_patients_nearby (≤40mm, high-baseline) 
# AND rebound_filtered_nostimsz (rebound data, stim-sz excluded).

# ── Step 1: Find shared patients ──────────────────────────────────────────────
shared_pts = (
    set(valid_patients_nearby['subject_id']) &
    set(rebound_filtered_nostimsz['subject_id'])
)
print(f"Patients in nearby (pre/during) plot : {valid_patients_nearby['subject_id'].nunique()}")
print(f"Patients in rebound (no-stim-sz)     : {rebound_filtered_nostimsz['subject_id'].nunique()}")
print(f"Shared patients (intersection)        : {len(shared_pts)}")

# ── Step 2: Pre & During — per-patient mean across all channels/trials ─────────
pt_pre_during = (
    valid_patients_nearby[valid_patients_nearby['subject_id'].isin(shared_pts)]
    [['subject_id', 'spike_rate_pre', 'spike_rate_during']]
    .copy()
)

# ── Step 3: 0–60s post — per-patient mean across all channels/bins ─────────────
post_0_60 = (
    rebound_filtered_nostimsz[
        rebound_filtered_nostimsz['subject_id'].isin(shared_pts) &
        (rebound_filtered_nostimsz['bin_start_s'] >= 30) &
        (rebound_filtered_nostimsz['bin_start_s'] < 90)
    ]
    .groupby('subject_id')['spikes_per_min']
    .mean()
    .rename('spike_rate_post_0_60')
    .reset_index()
)

# ── Step 4: 60–120s post — per-patient mean across all channels/bins ───────────
post_60_120 = (
    rebound_filtered_nostimsz[
        rebound_filtered_nostimsz['subject_id'].isin(shared_pts) &
        (rebound_filtered_nostimsz['bin_start_s'] >= 90) &
        (rebound_filtered_nostimsz['bin_start_s'] < 150)
    ]
    .groupby('subject_id')['spikes_per_min']
    .mean()
    .rename('spike_rate_post_60_120')
    .reset_index()
)

# ── Step 5: Merge all four timepoints ─────────────────────────────────────────
combined_4pt = (
    pt_pre_during
    .merge(post_0_60,   on='subject_id', how='inner')
    .merge(post_60_120, on='subject_id', how='inner')
    .dropna(subset=['spike_rate_pre', 'spike_rate_during',
                    'spike_rate_post_0_60', 'spike_rate_post_60_120'])
)
print(f"\nFinal n = {len(combined_4pt)} patients in combined plot")

# ── Step 6: Plot ───────────────────────────────────────────────────────────────
x_labels  = ['Pre-\nStimulation', 'During\nStimulation', '0–60s\nPost', '60–120s\nPost']
col_keys  = ['spike_rate_pre', 'spike_rate_during', 'spike_rate_post_0_60', 'spike_rate_post_60_120']
pt_colors = [COLORS['yellow'], COLORS['teal'], '#C8835C', COLORS['brown']]
x = [0, 1, 2, 3]

# Adjacent segment pairs: (left index, right index)
segment_pairs = [(0, 1), (1, 2), (2, 3)]

fig, ax = plt.subplots(1, 1, figsize=(70 * MM_TO_IN, 51 * MM_TO_IN))

# Color-coded patient lines — each segment takes the color of the higher endpoint
for _, row in combined_4pt.iterrows():
    y = [row[c] for c in col_keys]
    for i, j in segment_pairs:
        seg_color = pt_colors[j] if y[j] > y[i] else pt_colors[i]
        ax.plot([x[i], x[j]], [y[i], y[j]],
                color=seg_color, alpha=0.35, linewidth=0.7)

# Medians
medians = [combined_4pt[c].median() for c in col_keys]
# ax.plot(x, medians, 'k-', linewidth=2, label='Median', zorder=11)

# Scatter points
for xi, (col, c) in enumerate(zip(col_keys, pt_colors)):
    ax.scatter(
        [xi] * len(combined_4pt), combined_4pt[col],
        color=c, s=30, zorder=10, edgecolor='black', linewidth=0.3, alpha=0.85
    )

# Symlog y-axis
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=6)
ax.set_ylabel('Spike Rate (spikes/min; log scale)')
ax.set_xlim(-0.3, 3.3)

# ── Step 7: Statistical annotations (Wilcoxon, Bonferroni) ────────────────────
paired_data_4pt = []
for _, row in combined_4pt.iterrows():
    for tp, col in zip(x_labels, col_keys):
        paired_data_4pt.append({
            'subject_id': row['subject_id'],
            'Timepoint': tp,
            'Spike Rate': row[col]
        })
paired_df_4pt = pd.DataFrame(paired_data_4pt)

pairs_4pt = [
    (x_labels[0], x_labels[1]),   # Pre vs During
    (x_labels[1], x_labels[2]),   # During vs 0–60s
    (x_labels[1], x_labels[3]),   # During vs 60–120s
    (x_labels[2], x_labels[3]),   # 0–60s vs 60–120s
    (x_labels[0], x_labels[2]),   # Pre vs 0–60s
    (x_labels[0], x_labels[3]),   # Pre vs 60–120s
]
annotator = Annotator(ax, pairs_4pt, data=paired_df_4pt, x='Timepoint', y='Spike Rate', order=x_labels)
annotator.configure(
    test='Wilcoxon',
    text_format='star',
    loc='inside',
    comparisons_correction='bonf',
    hide_non_significant=True,
    verbose=True
)
annotator.apply_and_annotate()

n_pts = len(combined_4pt)
# ax.set_title(f'n = {n_pts} patients', fontsize=7, pad=4)
plt.show()

# ── Summary statistics ─────────────────────────────────────────────────────────
print("\n=== SUMMARY STATISTICS ===")
for label, col in zip(x_labels, col_keys):
    arr = combined_4pt[col]
    label_clean = label.replace('\n', ' ')
    print(f"{label_clean:22}  Median: {arr.median():.2f}  IQR: [{arr.quantile(0.25):.2f} – {arr.quantile(0.75):.2f}]  n={arr.notna().sum()}")

In [ ]:
col_keys = ['spike_rate_pre', 'spike_rate_during', 'spike_rate_post_0_60', 'spike_rate_post_60_120']
labels   = ['Pre-Stim', 'During-Stim', '30–90s Post', '90–150s Post']
pairs_idx = [(0,1),(1,2),(1,3),(2,3),(0,2),(0,3)]
print(f"{'Comparison':<38} {'δ':>7}  {'|δ|':>6}  Magnitude")
print("─" * 65)
for i, j in pairs_idx:
    d   = cliffs_delta(combined_4pt[col_keys[i]], combined_4pt[col_keys[j]])
    mag = interpret_cliffs_delta(d)
    cmp = f"{labels[i]}  vs  {labels[j]}"
    print(f"{cmp:<38} {d:>+7.3f}  {abs(d):>6.3f}  {mag}")
print(f"\nn = {len(combined_4pt)} patients")

# Figure 2

In [ ]:
analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_inter.csv'))
print(f'Loaded inter-stim data: {analysis_data.shape}')


## Change by ROI

In [ ]:
# Define brain region categories for STIMULATION sites
MTL_STIM_PARCELS = [
    "Left-Hippocampus", "Right-Hippocampus",
    "Left-Amygdala", "Right-Amygdala",
    "ctx-lh-entorhinal",    "ctx-rh-entorhinal",
    "ctx-lh-parahippocampal", "ctx-rh-parahippocampal",
    "ctx-lh-perirhinal", "ctx-rh-perirhinal"
]

TEMPORAL_NEOCORTICAL_STIM_PARCELS = [
    "ctx-lh-superiortemporal", "ctx-rh-superiortemporal",
    "ctx-lh-middletemporal", "ctx-rh-middletemporal", 
    "ctx-lh-inferiortemporal", "ctx-rh-inferiortemporal",
    "ctx-lh-transversetemporal", "ctx-rh-transversetemporal",
    "ctx-lh-temporalpole", "ctx-rh-temporalpole",
    "ctx-lh-fusiform", "ctx-rh-fusiform"
]

WHITE_MATTER_STIM_PARCELS = [
    "white-matter", "wm-lh-", "wm-rh-", "cerebellum-white-matter"
]

def categorize_stimulation_region(roi):
    if pd.isna(roi):
        return 'Other Cortex'
    if roi in MTL_STIM_PARCELS:
        return 'MTL'
    elif roi in TEMPORAL_NEOCORTICAL_STIM_PARCELS: 
        return 'Temporal Neocortex'
    else:
        return 'Other Cortex'

# ----------------------------------------------------------------
# Create high_baseline_set for efficient lookup
# ----------------------------------------------------------------
# high_baseline_set = set()
# for subject_id, channels in high_baseline_chs.items():
#     for ch_dict in channels:
#         high_baseline_set.add((subject_id, ch_dict['recording_ch']))

stim_regions = ['MTL', 'Temporal Neocortex', 'Other Cortex', 'White Matter']
region_colors = [COLORS['teal'], COLORS['yellow'], COLORS['brown'], COLORS['gray']]

# ----------------------------------------------------------------
# Loop through both SOZ and High Spiking channel analyses
# ----------------------------------------------------------------
for analysis_type in ['soz']:
    
    if analysis_type == 'soz':
        nearby_data = analysis_data[
            (analysis_data['distance_mm'] <= 40) 
        ].copy()
        title_text = 'Δ Spike Rate by ROI'
        filename = 'during_to_inter_stim_region_nearby_soz.pdf'
    else:
        nearby_data = analysis_data[
            (analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1))
        ].copy()
        title_text = 'Δ Spike Rate by ROI (Rec: IZ, Stim: in ROI)'
        filename = 'during_to_inter_stim_region_nearby_high_spiking.pdf'
    
    print(f"\n{'='*70}")
    print(f"Analysis: {analysis_type}")
    print(f"Total observations after filtering: {len(nearby_data)}")
    print(f"{'='*70}")
    
    if len(nearby_data) == 0:
        print("No data available for this analysis version.")
        continue
    
    nearby_data['stim_region_category'] = nearby_data['roi_stim_ch'].apply(categorize_stimulation_region)
    
    change_col = 'change_during_to_inter'
    
    patient_averages = []
    for region in stim_regions:
        region_data = nearby_data[nearby_data['stim_region_category'] == region]
        if len(region_data) > 0:
            patient_avg_change = region_data.groupby('subject_id')[change_col].mean()
            patient_averages.append(patient_avg_change.dropna())
        else:
            patient_averages.append(pd.Series([], dtype=float))
    
    fig, ax = plt.subplots(1, 1, figsize=(60 * MM_TO_IN, 60 * MM_TO_IN))

    for i, (region, patient_data, color) in enumerate(zip(stim_regions, patient_averages, region_colors)):
        if len(patient_data) > 0:
            x_jitter = np.random.normal(i, 0.05, size=len(patient_data))
            ax.scatter(x_jitter, patient_data, alpha=0.7, s=25, color=color,
                    edgecolors='black', linewidth=0.3, zorder=3)
            
            median_value = patient_data.median()
            ax.plot([i-0.3, i+0.3], [median_value, median_value],
                   color='black', linewidth=2, zorder=4)

    ax.set_yscale('symlog', linthresh=1)
    ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1000, -100, -10, -1, 0, 1, 10, 100, 1000]))

    ax.set_xticks(range(len(stim_regions)))
    ax.set_xticklabels(['MTL', 'TN', 'OC', 'WM'])
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.7, linewidth=1)
    ax.set_ylabel('Δ Spike Rate (spike/min; symlog)')
    # ax.set_xlabel('Stimulation ROI')
    # ax.set_title(title_text)
    plt.xticks(rotation=0, ha='center', fontsize = 6)

    plot_data = []
    for i, (region, patient_data) in enumerate(zip(stim_regions, patient_averages)):
        if len(patient_data) > 0:
            for value in patient_data:
                plot_data.append({'change': value, 'region': region})
    
    plot_df = pd.DataFrame(plot_data)
    
    valid_regions = [region for region, data in zip(stim_regions, patient_averages) if len(data) > 0]
    pairs = []
    for i in range(len(valid_regions)):
        for j in range(i+1, len(valid_regions)):
            pairs.append((valid_regions[i], valid_regions[j]))
    
    if len(pairs) > 0 and len(plot_df) > 0:
        annotator = Annotator(
            ax, 
            pairs, 
            data=plot_df, 
            x='region', 
            y='change',
            order=valid_regions
        )
        annotator.configure(
            test='Mann-Whitney', 
            text_format='star', 
            loc='inside',
            verbose=True,
            comparisons_correction="bonf",
            hide_non_significant=True
        )
        annotator.apply_and_annotate()

    valid_groups = [group for group in patient_averages if len(group) > 0]
    valid_group_names = [region for region, group in zip(stim_regions, patient_averages) if len(group) > 0]

    if len(valid_groups) > 1:
        kruskal_stat, kruskal_p = kruskal(*valid_groups)
        ax.text(0.95, 0.05, f'K.W. p = {kruskal_p:.3f}', 
                transform=ax.transAxes, 
                fontsize=5, 
                verticalalignment='bottom', 
                horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    # plt.savefig(ospj(CONFIG.fig_dir, filename), bbox_inches='tight', dpi=300)
    plt.show()

    print(f"\nPATIENT-LEVEL AVERAGE Δ SUMMARY BY STIMULATION LOCATION")
    print(f"Analysis: {analysis_type}")
    print(f"{'='*70}")
    for region, patient_data in zip(stim_regions, patient_averages):
        if len(patient_data) > 0:
            print(f"\n{region} Stimulation:")
            print(f"  Number of patients: {len(patient_data)}")
            print(f"  Median avg Δ: {patient_data.median():.4f}")
            print(f"  25th percentile: {patient_data.quantile(0.25):.4f}")
            print(f"  75th percentile: {patient_data.quantile(0.75):.4f}")
            print(f"  Mean avg Δ: {patient_data.mean():.4f}")
            print(f"  Std deviation: {patient_data.std():.4f}")
            print(f"  Patient IDs: {sorted(patient_data.index.tolist())}")
        else:
            print(f"\n{region} Stimulation: No data available")

    if len(valid_groups) > 1:
        print(f"\n=== STATISTICAL COMPARISON ===")
        print(f"Kruskal-Wallis test across stimulation regions (patient averages):")
        print(f"Statistic: {kruskal_stat:.4f}")
        print(f"P-value: {kruskal_p:.6f}")
        print(f"Groups compared: {valid_group_names}")
    else:
        print(f"\n=== STATISTICAL COMPARISON ===")
        print("Insufficient groups for statistical comparison")

    print(f"{'Pair':<38} {'Cliff d':>9}  {'Magnitude':>10}")
    print("-" * 62)

    valid = [(region, data) for region, data in zip(stim_regions, patient_averages) if len(data) > 0]

    for (r1, d1), (r2, d2) in itertools.combinations(valid, 2):
        d = cliffs_delta(d1.values, d2.values)
        mag = 'negligible' if abs(d) < 0.147 else 'small' if abs(d) < 0.33 else 'medium' if abs(d) < 0.474 else 'large'
        print(f"{r1} vs {r2:<20} {d:>9.3f}  {mag:>10}")

## During/inter by ROI

In [ ]:
stim_regions  = ['MTL', 'Temporal Neocortex', 'Other Cortex']
stim_labels   = ['MTL', 'TN', 'OC']
inter_color   = '#704929'
during_color  = '#E7CE95'

nearby_data = analysis_data[analysis_data['distance_mm'] <= 40].copy()
nearby_data['stim_region_category'] = nearby_data['roi_stim_ch'].apply(categorize_stimulation_region)

print(f"Total observations (≤40mm): {len(nearby_data)}")

inter_medians, during_medians = [], []
inter_sems, during_sems       = [], []
patient_dfs                   = []

for region in stim_regions:
    region_data = nearby_data[nearby_data['stim_region_category'] == region]
    if len(region_data) > 0:
        patient_avg = region_data.groupby('subject_id').agg({
            'spike_rate_inter':  'mean',
            'spike_rate_during': 'mean'
        }).dropna()
    else:
        patient_avg = pd.DataFrame(columns=['spike_rate_inter', 'spike_rate_during'])

    patient_dfs.append(patient_avg)

    inter_vals  = patient_avg['spike_rate_inter'].dropna()
    during_vals = patient_avg['spike_rate_during'].dropna()

    inter_medians.append(inter_vals.median()  if len(inter_vals)  > 0 else 0)
    during_medians.append(during_vals.median() if len(during_vals) > 0 else 0)
    inter_sems.append(inter_vals.sem()   if len(inter_vals)  > 1 else 0)
    during_sems.append(during_vals.sem() if len(during_vals) > 1 else 0)

# ----------------------------------------------------------------
# Plot
# ----------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(80 * MM_TO_IN, 60 * MM_TO_IN))

x     = np.arange(len(stim_regions))
width = 0.35

ax.bar(x - width/2, inter_medians,  width, label='Inter Stim',
       color=inter_color,  edgecolor='black', linewidth=0.5,
       yerr=inter_sems,  capsize=3, error_kw={'linewidth': 1, 'color': 'red'})
ax.bar(x + width/2, during_medians, width, label='During Stim',
       color=during_color, edgecolor='black', linewidth=0.5,
       yerr=during_sems, capsize=3, error_kw={'linewidth': 1, 'color': 'red'})

for i, patient_avg in enumerate(patient_dfs):
    if len(patient_avg) == 0:
        continue

    inter_vals  = patient_avg['spike_rate_inter'].dropna().values
    during_vals = patient_avg['spike_rate_during'].dropna().values

    x_jitter_inter  = np.random.normal(x[i] - width/2, 0.04, size=len(inter_vals))
    x_jitter_during = np.random.normal(x[i] + width/2, 0.04, size=len(during_vals))

    ax.scatter(x_jitter_inter,  inter_vals,  alpha=0.6, s=15, color=COLORS['teal'],
               edgecolors='black', linewidth=0.3, zorder=5)
    ax.scatter(x_jitter_during, during_vals, alpha=0.6, s=15, color=COLORS['teal'],
               edgecolors='black', linewidth=0.3, zorder=5)

ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))

ax.set_xticks(x)
ax.set_xticklabels(stim_labels, fontsize = 6)
ax.set_ylabel('Spike Rate (spikes/min; log scale)')
# ax.set_xlabel('Stimulation ROI')
ax.legend(loc='upper right', fontsize=7)
# ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
# plt.savefig(ospj(CONFIG.fig_dir, 'grouped_bar_inter_during_by_roi.pdf'), bbox_inches='tight', dpi=300)
plt.show()

# ----------------------------------------------------------------
# Summary statistics and within-ROI comparisons
# ----------------------------------------------------------------
alpha: float = 0.05
n_comparisons: int = 3
print(f"\n{'='*70}")
print("Within-ROI inter vs during (Wilcoxon signed-rank, "
      f"Bonferroni n={n_comparisons})")
print(f"{'='*70}")
within_rows: list[dict[str, float | str | int]] = []
for region, label, patient_avg in zip(stim_regions, stim_labels, patient_dfs):
    if len(patient_avg) == 0:
        print(f"\n{label}: No data")
        continue
    inter_vals  = patient_avg['spike_rate_inter'].dropna()
    during_vals = patient_avg['spike_rate_during'].dropna()
    print(f"\n{region} ({label})  n={len(patient_avg)}")
    print(f"  Inter  — Median: {inter_vals.median():.2f}, "
          f"Mean: {inter_vals.mean():.2f}")
    print(f"  During — Median: {during_vals.median():.2f}, "
          f"Mean: {during_vals.mean():.2f}")
    if len(patient_avg) > 1:
        stat, p_raw = wilcoxon(inter_vals, during_vals, alternative='two-sided')
        p_bonf = min(p_raw * n_comparisons, 1.0)
        sig = "*" if p_bonf < alpha else "ns"
        print(f"  Wilcoxon inter vs during: W = {stat:.2f}, "
              f"p_raw = {p_raw:.4f}, p_bonf = {p_bonf:.4f}  [{sig}]")
        within_rows.append({
            "region": label,
            "n": int(len(patient_avg)),
            "W": float(stat),
            "p_raw": float(p_raw),
            "p_bonf": float(p_bonf),
        })
    else:
        print("  Not enough data for Wilcoxon.")


In [ ]:
inter_by_region: dict[str, np.ndarray] = {}
for label, patient_avg in zip(stim_labels, patient_dfs):
    vals = patient_avg['spike_rate_inter'].dropna().values
    inter_by_region[label] = vals
    print(f"{label}: n={len(vals)}, median={np.median(vals):.3f}, "
          f"IQR=({np.quantile(vals, 0.25):.3f} - {np.quantile(vals, 0.75):.3f})")

groups_with_data = [v for v in inter_by_region.values() if len(v) > 0]
labels_with_data = [k for k, v in inter_by_region.items() if len(v) > 0]

print("\n=== Kruskal–Wallis (inter-stim spike rate across ROIs) ===")
if len(groups_with_data) >= 2 and all(len(v) > 1 for v in groups_with_data):
    H, p_kw = kruskal(*groups_with_data)
    print(f"H = {H:.3f}, p = {p_kw:.4f}  (groups: {labels_with_data})")
else:
    H, p_kw = np.nan, np.nan
    print("Not enough data in >=2 groups to run Kruskal–Wallis.")

alpha: float = 0.05
n_comparisons: int = 3
print(f"\n=== Pairwise Mann–Whitney U with Bonferroni (n_comp={n_comparisons}) ===")
justified = (not np.isnan(p_kw)) and (p_kw < alpha)
if not justified:
    print(f"Note: KW p = {p_kw:.4f} >= {alpha}; pairwise tests below are "
          f"exploratory only (post-hocs not justified by the omnibus).")

pairwise_rows: list[dict[str, float | str]] = []
for a, b in combinations(labels_with_data, 2):
    va, vb = inter_by_region[a], inter_by_region[b]
    if len(va) < 2 or len(vb) < 2:
        print(f"  {a} vs {b}: insufficient data")
        continue
    U, p_raw = mannwhitneyu(va, vb, alternative='two-sided')
    p_bonf = min(p_raw * n_comparisons, 1.0)
    sig = "*" if p_bonf < alpha else "ns"
    print(f"  {a} (n={len(va)}) vs {b} (n={len(vb)}): "
          f"U = {U:.1f}, p_raw = {p_raw:.4f}, p_bonf = {p_bonf:.4f}  [{sig}]")
    pairwise_rows.append({
        "pair": f"{a} vs {b}",
        "n_a": len(va), "n_b": len(vb),
        "U": U, "p_raw": p_raw, "p_bonf": p_bonf,
    })

pairwise_inter_df = pd.DataFrame(pairwise_rows)
pairwise_inter_df

## During/inter - by MTL & MTLE

### reload analysis_data

In [ ]:
analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_inter.csv'))
print(f'Loaded inter-stim data: {analysis_data.shape}')


### ipsi

In [ ]:
# MTL stimulation regions
MTL_REGIONS = {
    'Left-Hippocampus', 'Right-Hippocampus',
    'Left-Amygdala', 'Right-Amygdala',
    'ctx-lh-entorhinal', 'ctx-rh-entorhinal',
    'ctx-lh-parahippocampal', 'ctx-rh-parahippocampal',
    "ctx-lh-perirhinal", "ctx-rh-perirhinal"
}

LEFT_MTL_REGIONS = {
    'Left-Hippocampus',
    'Left-Amygdala',
    'ctx-lh-entorhinal',
    'ctx-lh-parahippocampal',
    "ctx-lh-perirhinal"
}
RIGHT_MTL_REGIONS = {
    'Right-Hippocampus',
    'Right-Amygdala',
    'ctx-rh-entorhinal',
    'ctx-rh-parahippocampal',
    "ctx-rh-perirhinal"
}

In [ ]:
metadata = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {metadata.shape}')

# ----------------------------------------------------------------
# Create high_baseline_set for efficient lookup
# ----------------------------------------------------------------
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))
# Filter: recording channel is nearby (≤30mm) + in high-baseline set + stimulation is in MTL
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40) &
    (analysis_data['roi_stim_ch'].isin(MTL_REGIONS))
    #  &
    # (analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1))
].copy()
# Derive stim side from roi_stim_ch
nearby_data['stim_side'] = nearby_data['roi_stim_ch'].apply(
    lambda r: 'left' if r in LEFT_MTL_REGIONS else 'right'
)
# Merge lateralization from metadata
nearby_data = nearby_data.merge(
    metadata[['subject_id', 'lateralization', 'EC_any_mtl']],
    on='subject_id', how='left'
)
# # Keep only rows where stim side matches patient's MTL lateralization
# # bilateral patients match both sides
nearby_data = nearby_data[
    (nearby_data['lateralization'].str.lower() == 'bilateral') |
    (nearby_data['stim_side'] == nearby_data['lateralization'].str.lower())
].copy()
print(f"\n{'='*80}")
print(f"Analysis: ipsilateral to side of epilepsy + Nearby Recording (≤40mm) + MTL Stimulation")
print(f"Total observations after filtering: {len(nearby_data)}")
print(f"{'='*80}")
if len(nearby_data) > 0:
    change_col = 'change_during_to_inter'
    stim_rates = nearby_data.groupby('subject_id', as_index=False)[change_col].mean()
    stim_rates.rename(columns={change_col: 'change'}, inplace=True)
    stim_rates = stim_rates.merge(metadata, on='subject_id', how='left')
    # Binary MTL categorization using EC_only_mtl
    stim_rates['is_MTL'] = stim_rates['EC_any_mtl'].str.lower() == 'yes'
    stim_rates['group'] = stim_rates['is_MTL'].map({True: 'ANY MTL', False: 'NO MTL'})
    mtl_data = stim_rates[stim_rates['group'] == 'ANY MTL']['change'].dropna()
    not_mtl_data = stim_rates[stim_rates['group'] == 'NO MTL']['change'].dropna()
    fig, ax = plt.subplots(1, 1, figsize=(60 * MM_TO_IN, 60 * MM_TO_IN))
    groups = ['MTLE', 'OTHER']
    group_data = [mtl_data, not_mtl_data]
    group_colors = [COLORS['teal'], COLORS['yellow']]
    for i, (group, patient_data, color) in enumerate(zip(groups, group_data, group_colors)):
        if len(patient_data) > 0:
            x_jitter = np.random.normal(i, 0.05, size=len(patient_data))
            ax.scatter(x_jitter, patient_data, alpha=0.7, s=25, color=color,
                       edgecolors='black', linewidth=0.3, zorder=3)
            mean_val = patient_data.median()
            ax.plot([i-0.2, i+0.2], [mean_val, mean_val],
                    color='black', linewidth=2, zorder=4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(groups, rotation=0, ha='center')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.8, linewidth=1)
    ax.set_yscale('symlog', linthresh=1)
    ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1000, -100, -10, -1, 0, 1, 10, 100, 1000]))
    ax.set_ylabel('Δ Spike Rate (spike/min; symlog)')
    plt.xticks(rotation=0, ha='center', fontsize = 6)
    # ax.set_xlabel('Patient Group')
    # ax.set_title('During→Inter Δ (Rec: ≤40mm, Stim: MTL)')
    if len(mtl_data) > 0 and len(not_mtl_data) > 0:
        from scipy.stats import mannwhitneyu
        stat, p_value = mannwhitneyu(mtl_data, not_mtl_data, alternative='two-sided')
        # ax.text(0.05, 0.95, f'Mann-Whitney p = {p_value:.3f}',
        #         transform=ax.transAxes,
        #         fontsize=5,
        #         verticalalignment='top',
        #         horizontalalignment='left',
        #         bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    ax.margins(y=0.15)
    plt.show()
    print(f"\n=== SUMMARY: ONLY MTL vs NO MTL ===")
    print(f"\nANY MTL Patients:")
    print(f"  Number of patients: {len(mtl_data)}")
    print(f"  Median Δ: {mtl_data.median():.4f}")
    print(f"  25th percentile Δ: {mtl_data.quantile(0.25):.4f}")
    print(f"  75th percentile Δ: {mtl_data.quantile(0.75):.4f}")
    print(f"  Mean Δ: {mtl_data.mean():.4f}")
    print(f"  Std deviation: {mtl_data.std():.4f}")
    print(f"  Patient IDs: {sorted(stim_rates[stim_rates['group'] == 'ANY MTL']['subject_id'].tolist())}")
    print(f"\nNO MTL Patients:")
    print(f"  Number of patients: {len(not_mtl_data)}")
    print(f"  Median Δ: {not_mtl_data.median():.4f}")
    print(f"  25th percentile Δ: {not_mtl_data.quantile(0.25):.4f}")
    print(f"  75th percentile Δ: {not_mtl_data.quantile(0.75):.4f}")
    print(f"  Mean Δ: {not_mtl_data.mean():.4f}")
    print(f"  Std deviation: {not_mtl_data.std():.4f}")
    print(f"  Patient IDs: {sorted(stim_rates[stim_rates['group'] == 'NO MTL']['subject_id'].tolist())}")
    if len(mtl_data) > 0 and len(not_mtl_data) > 0:
        print(f"\n=== STATISTICAL COMPARISON ===")
        print(f"Mann-Whitney U test (ONLY MTL vs NO MTL):")
        print(f"Statistic: {stat:.4f}")
        print(f"P-value: {p_value:.6f}")
        print(f"Effect: ONLY MTL median = {mtl_data.median():.4f}, NO MTL median = {not_mtl_data.median():.4f}")
    print("\n" + "="*80 + "\n")
else:
    print("No data available after filtering.")


In [ ]:
metadata = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {metadata.shape}')

# ----------------------------------------------------------------
# Create high_baseline_set for efficient lookup
# ----------------------------------------------------------------
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))
# Filter: recording channel is nearby (≤30mm) + in high-baseline set + stimulation is in MTL
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40) &
    (analysis_data['roi_stim_ch'].isin(MTL_REGIONS))
    #  &
    # (analysis_data.apply(lambda row: (row['subject_id'], row['recording_ch']) in high_baseline_set, axis=1))
].copy()
# Derive stim side from roi_stim_ch
nearby_data['stim_side'] = nearby_data['roi_stim_ch'].apply(
    lambda r: 'left' if r in LEFT_MTL_REGIONS else 'right'
)
# Merge lateralization from metadata
nearby_data = nearby_data.merge(
    metadata[['subject_id', 'lateralization', 'EC_any_mtl']],
    on='subject_id', how='left'
)
# Keep only rows where stim side matches patient's MTL lateralization
# bilateral patients match both sides
nearby_data = nearby_data[
    (nearby_data['lateralization'].str.lower() == 'bilateral') |
    (nearby_data['stim_side'] == nearby_data['lateralization'].str.lower())
].copy()
print(f"\n{'='*80}")
print(f"Analysis: High Spiking Channels + Nearby Recording (≤30mm) + MTL Stimulation")
print(f"Total observations after filtering: {len(nearby_data)}")
print(f"{'='*80}")
if len(nearby_data) > 0:
    patient_rates = nearby_data.groupby('subject_id', as_index=False).agg({
        'spike_rate_inter': 'mean',
        'spike_rate_during': 'mean'
    })
    patient_rates = patient_rates.merge(metadata, on='subject_id', how='left')
    # Binary MTL categorization using EC_any_mtl
    patient_rates['is_MTL'] = patient_rates['EC_any_mtl'].str.lower() == 'yes'
    patient_rates['group'] = patient_rates['is_MTL'].map({True: 'ANY MTL', False: 'NO MTL'})
    mtl_patients = patient_rates[patient_rates['group'] == 'ANY MTL']
    not_mtl_patients = patient_rates[patient_rates['group'] == 'NO MTL']
    groups = ['MTLE', 'OTHER']
    group_dfs = [mtl_patients, not_mtl_patients]
    inter_medians, during_medians = [], []
    inter_sems, during_sems = [], []
    for gdf in group_dfs:
        inter_vals = gdf['spike_rate_inter'].dropna()
        during_vals = gdf['spike_rate_during'].dropna()
        inter_medians.append(inter_vals.median() if len(inter_vals) > 0 else 0)
        during_medians.append(during_vals.median() if len(during_vals) > 0 else 0)
        inter_sems.append(inter_vals.sem() if len(inter_vals) > 1 else 0)
        during_sems.append(during_vals.sem() if len(during_vals) > 1 else 0)
    fig, ax = plt.subplots(1, 1, figsize=(80 * MM_TO_IN, 60 * MM_TO_IN))
    x = np.arange(len(groups))
    width = 0.35
    inter_color = '#704929'
    during_color = '#E7CE95'
    bars_inter = ax.bar(x - width/2, inter_medians, width, label='Inter-Stim',
                        color=inter_color, edgecolor='black', linewidth=0.5,
                        yerr=inter_sems, capsize=3, error_kw={'linewidth': 1, 'color': 'red'})
    bars_during = ax.bar(x + width/2, during_medians, width, label='During Stim',
                         color=during_color, edgecolor='black', linewidth=0.5,
                         yerr=during_sems, capsize=3, error_kw={'linewidth': 1, 'color': 'red'})
    for i, gdf in enumerate(group_dfs):
        inter_vals = gdf['spike_rate_inter'].dropna().values
        x_jitter_inter = np.random.normal(x[i] - width/2, 0.05, size=len(inter_vals))
        ax.scatter(x_jitter_inter, inter_vals, alpha=0.6, s=15, color=COLORS['teal'],
                   edgecolors='black', linewidth=0.3, zorder=5)
        during_vals = gdf['spike_rate_during'].dropna().values
        x_jitter_during = np.random.normal(x[i] + width/2, 0.05, size=len(during_vals))
        ax.scatter(x_jitter_during, during_vals, alpha=0.6, s=15, color=COLORS['teal'],
                   edgecolors='black', linewidth=0.3, zorder=5)
    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    ax.set_xlabel('')
    # ax.set_title('During→Inter Δ (Rec: IZ ≤30mm, Stim: MTL)')
    ax.legend(loc='upper right', fontsize=7)
    all_vals = np.concatenate([
        mtl_patients['spike_rate_inter'].dropna().values,
        mtl_patients['spike_rate_during'].dropna().values,
        not_mtl_patients['spike_rate_inter'].dropna().values,
        not_mtl_patients['spike_rate_during'].dropna().values
    ])
    ax.set_ylim(bottom=0, top=all_vals.max() * 1.5)
    ax.set_yscale('symlog', linthresh=1)
    ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))
    ax.set_ylabel('Spike Rate (spikes/min; log scale)')
    ax.legend(loc='upper right', fontsize=7)
    plt.tight_layout()
    plt.xticks(rotation=0, ha='center', fontsize = 6)
    plt.show()
    print(f"\n=== SUMMARY: ANY MTL vs NO MTL ===")
    for group_name, gdf in zip(groups, group_dfs):
        print(f"\n{group_name} Patients (n={len(gdf)}):")
        inter_median = gdf['spike_rate_inter'].median()
        inter_mean = gdf['spike_rate_inter'].mean()
        inter_q25 = gdf['spike_rate_inter'].quantile(0.25)
        inter_q75 = gdf['spike_rate_inter'].quantile(0.75)
        print(f"  Inter-Stim  - Median: {inter_median:.2f}, Mean: {inter_mean:.2f}, 25th percentile: {inter_q25:.2f}, 75th percentile: {inter_q75:.2f}")
        during_median = gdf['spike_rate_during'].median()
        during_mean = gdf['spike_rate_during'].mean()
        during_q25 = gdf['spike_rate_during'].quantile(0.25)
        during_q75 = gdf['spike_rate_during'].quantile(0.75)
        print(f"  During Stim - Median: {during_median:.2f}, Mean: {during_mean:.2f}, 25th percentile: {during_q25:.2f}, 75th percentile: {during_q75:.2f}")
        print(f"  Patient IDs: {sorted(gdf['subject_id'].tolist())}")
    from scipy.stats import wilcoxon
    print(f"\n=== WITHIN-GROUP COMPARISONS (Wilcoxon signed-rank) ===")
    for group_name, gdf in zip(groups, group_dfs):
        valid = gdf.dropna(subset=['spike_rate_inter', 'spike_rate_during'])
        if len(valid) > 5:
            stat, p = wilcoxon(valid['spike_rate_inter'], valid['spike_rate_during'])
            print(f"{group_name}: p = {p:.4f} (n={len(valid)})")
    print("\n" + "="*80 + "\n")
else:
    print("No data available after filtering.")


In [ ]:
alpha: float = 0.05
primary_method: str = 'bonferroni'
print("\n=== WITHIN-GROUP COMPARISONS (Wilcoxon signed-rank) ===")
within_rows: list[dict[str, float | str | int]] = []
for group_name, gdf in zip(groups, group_dfs):
    valid = gdf.dropna(subset=['spike_rate_inter', 'spike_rate_during'])
    n_valid: int = len(valid)
    if n_valid <= 5:
        print(f"{group_name}: n={n_valid} — too few pairs for Wilcoxon")
        continue
    stat, p_raw = wilcoxon(
        valid['spike_rate_inter'],
        valid['spike_rate_during'],
        alternative='two-sided',
    )
    within_rows.append({
        "group": group_name,
        "n": n_valid,
        "W": float(stat),
        "p_raw": float(p_raw),
    })
p_raws: list[float] = [r["p_raw"] for r in within_rows]
if len(p_raws) > 0:
    _, p_bonf, _, _ = multipletests(p_raws, alpha=alpha, method='bonferroni')
    _, p_holm, _, _ = multipletests(p_raws, alpha=alpha, method='holm')
    _, p_bh,   _, _ = multipletests(p_raws, alpha=alpha, method='fdr_bh')
    primary_map = {'bonferroni': p_bonf, 'holm': p_holm, 'fdr_bh': p_bh}
    p_primary = primary_map[primary_method]
    for row, pb, ph, pbh, pp in zip(within_rows, p_bonf, p_holm, p_bh, p_primary):
        row["p_bonf"] = float(pb)
        row["p_holm"] = float(ph)
        row["p_bh"]   = float(pbh)
        sig = "*" if pp < alpha else "ns"
        print(f"{row['group']} (n={row['n']}): W = {row['W']:.2f}, "
              f"p_raw = {row['p_raw']:.4f}, p_bonf = {pb:.4f}, "
              f"p_holm = {ph:.4f}, p_bh = {pbh:.4f}  "
              f"[{sig} @ {primary_method}]")
within_mtl_vs_other_df = pd.DataFrame(within_rows)
print("\nSummary table:")
within_mtl_vs_other_df


In [ ]:
print("\n=== BETWEEN-GROUP COMPARISONS (Mann-Whitney U) ===")
# Assumes `groups` has two entries, e.g. ['MTLE', 'non-MTLE'], and `group_dfs` is aligned.
g1_name, g2_name = groups[0], groups[1]
g1_df, g2_df = group_dfs[0], group_dfs[1]

between_rows: list[dict[str, float | str | int]] = []
for phase in ['spike_rate_inter', 'spike_rate_during']:
    x = g1_df[phase].dropna()
    y = g2_df[phase].dropna()
    n1, n2 = len(x), len(y)
    if n1 <= 5 or n2 <= 5:
        print(f"{phase}: n1={n1}, n2={n2} — too few samples for Mann-Whitney")
        continue
    stat, p_raw = mannwhitneyu(x, y, alternative='two-sided')
    between_rows.append({
        "comparison": f"{g1_name} vs {g2_name}",
        "phase": phase.replace('spike_rate_', ''),
        "n1": n1,
        "n2": n2,
        "median_1": float(np.median(x)),
        "median_2": float(np.median(y)),
        "U": float(stat),
        "p_raw": float(p_raw),
    })

p_raws_b: list[float] = [r["p_raw"] for r in between_rows]
if len(p_raws_b) > 0:
    _, p_bonf_b, _, _ = multipletests(p_raws_b, alpha=alpha, method='bonferroni')
    _, p_holm_b, _, _ = multipletests(p_raws_b, alpha=alpha, method='holm')
    _, p_bh_b,   _, _ = multipletests(p_raws_b, alpha=alpha, method='fdr_bh')
    primary_map_b = {'bonferroni': p_bonf_b, 'holm': p_holm_b, 'fdr_bh': p_bh_b}
    p_primary_b = primary_map_b[primary_method]
    for row, pb, ph, pbh, pp in zip(between_rows, p_bonf_b, p_holm_b, p_bh_b, p_primary_b):
        row["p_bonf"] = float(pb)
        row["p_holm"] = float(ph)
        row["p_bh"]   = float(pbh)
        sig = "*" if pp < alpha else "ns"
        print(f"{row['comparison']} [{row['phase']}] "
              f"(n1={row['n1']}, n2={row['n2']}): "
              f"U = {row['U']:.2f}, "
              f"median_{g1_name}={row['median_1']:.3f}, "
              f"median_{g2_name}={row['median_2']:.3f}, "
              f"p_raw = {row['p_raw']:.4f}, p_bonf = {pb:.4f}, "
              f"p_holm = {ph:.4f}, p_bh = {pbh:.4f}  "
              f"[{sig} @ {primary_method}]")

between_mtl_vs_other_df = pd.DataFrame(between_rows)
print("\nSummary table:")
between_mtl_vs_other_df

# Figure 3 (now part of figure 2)

## reload analysis data

In [ ]:
analysis_data = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_inter.csv'))
print(f'Loaded inter-stim data: {analysis_data.shape}')


In [ ]:
# Define brain region categories for STIMULATION sites
MTL_STIM_PARCELS = [
    "Left-Hippocampus", "Right-Hippocampus",
    "Left-Amygdala", "Right-Amygdala",
    "ctx-lh-entorhinal",    "ctx-rh-entorhinal",
    "ctx-lh-parahippocampal", "ctx-rh-parahippocampal",
    "ctx-lh-perirhinal", "ctx-rh-perirhinal"
]

TEMPORAL_NEOCORTICAL_STIM_PARCELS = [
    "ctx-lh-superiortemporal", "ctx-rh-superiortemporal",
    "ctx-lh-middletemporal", "ctx-rh-middletemporal", 
    "ctx-lh-inferiortemporal", "ctx-rh-inferiortemporal",
    "ctx-lh-transversetemporal", "ctx-rh-transversetemporal",
    "ctx-lh-temporalpole", "ctx-rh-temporalpole",
    "ctx-lh-fusiform", "ctx-rh-fusiform"
]

WHITE_MATTER_STIM_PARCELS = [
    "white-matter", "wm-lh-", "wm-rh-", "cerebellum-white-matter"
]

def categorize_stimulation_region(roi):
    if pd.isna(roi):
        return 'Other Cortex'
    if roi in MTL_STIM_PARCELS:
        return 'MTL'
    elif roi in TEMPORAL_NEOCORTICAL_STIM_PARCELS: 
        return 'Temporal Neocortex'
    else:
        return 'Other Cortex'

## change

In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin


In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin['stim_sz_mtl'] = stimsz_erin['Stim seizure elicited in MTL'].map({'Yes': 1, 'No': 0})
stimsz_erin = stimsz_erin[['subject_id', 'stim_sz_mtl']].dropna(subset=['stim_sz_mtl'])
stimsz_erin['stim_sz_mtl'] = stimsz_erin['stim_sz_mtl'].astype(int)
print(f"Patients with MTL stim sz data: {len(stimsz_erin)}")
print(stimsz_erin['stim_sz_mtl'].value_counts().rename({0: 'No Stim Sz', 1: 'Stim Sz'}))
# Category labels and colors
# ----------------------------------------------------------------
category_order = ['No Stim Sz', 'Stim Sz']
category_colors = [COLORS['teal'], COLORS['yellow']]
# Analysis
# ----------------------------------------------------------------
#only close pairs
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40)
].copy()
title_str = 'Δ Spike Rate by MTL Stim Sz (Rec: 30mm, Stim: MTL)'
print(f"\n{'='*80}")
print(f"Analysis: {analysis_type}")
print(f"Observations before MTL stim filter: {len(nearby_data)}")
#keep only rows where the stimulation site is MTL
nearby_data['stim_region_category'] = nearby_data['roi_stim_ch'].apply(categorize_stimulation_region)
nearby_data = nearby_data[nearby_data['stim_region_category'] == 'MTL'].copy()
print(f"Observations after MTL stim filter:  {len(nearby_data)}")
print(f"{'='*80}")
if len(nearby_data) == 0:
    print("No data available for this analysis type.")
# average change per patient, then merge stim_sz_mtl label
change_col = 'change_during_to_inter'
patient_avg = nearby_data.groupby('subject_id', as_index=False)[change_col].mean()
patient_avg.rename(columns={change_col: 'change'}, inplace=True)
patient_avg = patient_avg.merge(stimsz_erin, on='subject_id', how='inner')
patient_avg['stim_sz_cat'] = patient_avg['stim_sz_mtl'].map({0: 'No Stim Sz', 1: 'Stim Sz'})
print(f"Patients after merging MTL stim sz label: {len(patient_avg)}")
print(patient_avg['stim_sz_cat'].value_counts())
# collect per-group data in category_order
existing_categories = [cat for cat in category_order if cat in patient_avg['stim_sz_cat'].values]
patient_averages = [
    patient_avg[patient_avg['stim_sz_cat'] == cat]['change'].dropna()
    for cat in existing_categories
]
group_colors = [category_colors[category_order.index(cat)] for cat in existing_categories]
# plot
fig, ax = plt.subplots(1, 1, figsize=(60 * MM_TO_IN, 60 * MM_TO_IN))
for i, (category, patient_data, color) in enumerate(zip(existing_categories, patient_averages, group_colors)):
    if len(patient_data) > 0:
        x_jitter = np.random.normal(i, 0.05, size=len(patient_data))
        ax.scatter(x_jitter, patient_data, alpha=0.7, s=25, color=color,
                    edgecolors='black', linewidth=0.3, zorder=3)
        median_val = patient_data.median()
        ax.plot([i - 0.15, i + 0.15], [median_val, median_val],
                color='black', linewidth=2, zorder=4)
ax.set_xticks(range(len(existing_categories)))
ax.set_xticklabels(existing_categories, rotation=0, ha='center', fontsize=6)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.8, linewidth=1)
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1000, -100, -10, -1, 0, 1, 10, 100, 1000]))
ax.set_ylabel('Δ Spike Rate [spike/min; symlog]')
# ax.set_title(title_str)
ax.margins(y=0.15)
# statannotations
plot_data = []
for category, patient_data in zip(existing_categories, patient_averages):
    for value in patient_data:
        plot_data.append({'change': value, 'category': category})
plot_df = pd.DataFrame(plot_data)
valid_categories = [cat for cat, data in zip(existing_categories, patient_averages) if len(data) > 0]
pairs = [(valid_categories[i], valid_categories[j])
            for i in range(len(valid_categories))
            for j in range(i + 1, len(valid_categories))]
if len(pairs) > 0 and len(plot_df) > 0:
    annotator = Annotator(ax, pairs, data=plot_df, x='category', y='change', order=valid_categories)
    annotator.configure(
        test='Mann-Whitney',
        text_format='star',
        loc='inside',
        verbose=True,
        comparisons_correction=None,
        hide_non_significant=True
    )
    annotator.apply_and_annotate()
plt.show()
# Statistical comparison
valid_groups      = [g for g in patient_averages if len(g) > 0]
valid_group_names = [cat for cat, g in zip(existing_categories, patient_averages) if len(g) > 0]
if len(valid_groups) == 2:
    mw_stat, mw_p = mannwhitneyu(valid_groups[0], valid_groups[1], alternative='two-sided')
    d = cliffs_delta(valid_groups[0].values, valid_groups[1].values)
    print(f"\n=== STATISTICAL COMPARISON ===")
    print(f"Groups: {valid_group_names}")
    print(f"Mann-Whitney U: statistic = {mw_stat:.4f}, p = {mw_p:.6f}")
    print(f"Cliff's d = {d:.3f}  (|d|: <0.147 negligible, <0.33 small, <0.474 medium, ≥0.474 large)")


In [ ]:
stimsz_erin = pd.read_csv(ospj(CONFIG.dataset_dir, 'subject_metadata.csv'))
print(f'Loaded subject metadata: {stimsz_erin.shape}')

stimsz_erin['stim_sz_mtl'] = stimsz_erin['Stim seizure elicited in MTL'].map({'Yes': 1, 'No': 0})
stimsz_erin = stimsz_erin[['subject_id', 'stim_sz_mtl']].dropna(subset=['stim_sz_mtl'])
stimsz_erin['stim_sz_mtl'] = stimsz_erin['stim_sz_mtl'].astype(int)
print(f"Patients with MTL stim sz data: {len(stimsz_erin)}")
print(stimsz_erin['stim_sz_mtl'].value_counts().rename({0: 'No Stim Sz', 1: 'Stim Sz'}))
# Build high_baseline_set
high_baseline_set = set()
for subject_id, channels in high_baseline_chs.items():
    for ch_dict in channels:
        high_baseline_set.add((subject_id, ch_dict['recording_ch']))
groups = ['No Stim Sz', 'Stim Sz']
inter_color = '#704929'
during_color = '#E7CE95'
nearby_data = analysis_data[
    (analysis_data['distance_mm'] <= 40) 
].copy()
# Step 2: restrict to MTL stimulation sites
nearby_data['stim_region_category'] = nearby_data['roi_stim_ch'].apply(categorize_stimulation_region)
nearby_data = nearby_data[nearby_data['stim_region_category'] == 'MTL'].copy()
print(f"Observations after MTL stim filter:  {len(nearby_data)}")
print(f"{'='*80}")
if len(nearby_data) == 0:
    print("No data available for this analysis type.")
# Step 3: average spike_rate_inter and spike_rate_during per patient, then merge label
patient_rates = nearby_data.groupby('subject_id', as_index=False).agg({
    'spike_rate_inter': 'mean',
    'spike_rate_during': 'mean'
})
patient_rates = patient_rates.merge(stimsz_erin, on='subject_id', how='inner')
patient_rates['group'] = patient_rates['stim_sz_mtl'].map({0: 'No Stim Sz', 1: 'Stim Sz'})
print(f"Patients after merging MTL stim sz label: {len(patient_rates)}")
print(patient_rates['group'].value_counts())
no_stim_sz_patients = patient_rates[patient_rates['group'] == 'No Stim Sz']
stim_sz_patients    = patient_rates[patient_rates['group'] == 'Stim Sz']
group_dfs = [no_stim_sz_patients, stim_sz_patients]
# Step 4: compute medians and SEMs for bar heights
inter_medians, during_medians = [], []
inter_sems, during_sems = [], []
for gdf in group_dfs:
    inter_vals  = gdf['spike_rate_inter'].dropna()
    during_vals = gdf['spike_rate_during'].dropna()
    inter_medians.append(inter_vals.median() if len(inter_vals) > 0 else 0)
    during_medians.append(during_vals.median() if len(during_vals) > 0 else 0)
    inter_sems.append(inter_vals.sem() if len(inter_vals) > 1 else 0)
    during_sems.append(during_vals.sem() if len(during_vals) > 1 else 0)
# Step 5: grouped bar plot
fig, ax = plt.subplots(1, 1, figsize=(80 * MM_TO_IN, 60 * MM_TO_IN))
x = np.arange(len(groups))
width = 0.35
bars_inter  = ax.bar(x - width/2, inter_medians,  width, label='Inter Stim',
                        color=inter_color,  edgecolor='black', linewidth=0.5,
                        yerr=inter_sems,  capsize=3, error_kw={'linewidth': 1, 'color': 'red'})
bars_during = ax.bar(x + width/2, during_medians, width, label='During Stim',
                        color=during_color, edgecolor='black', linewidth=0.5,
                        yerr=during_sems, capsize=3, error_kw={'linewidth': 1, 'color': 'red'})
# Overlay individual patient points
for i, gdf in enumerate(group_dfs):
    inter_vals  = gdf['spike_rate_inter'].dropna().values
    during_vals = gdf['spike_rate_during'].dropna().values
    x_jitter_inter  = np.random.normal(x[i] - width/2, 0.05, size=len(inter_vals))
    x_jitter_during = np.random.normal(x[i] + width/2, 0.05, size=len(during_vals))
    ax.scatter(x_jitter_inter,  inter_vals,  alpha=0.6, s=15, color=COLORS['teal'],
                edgecolors='black', linewidth=0.3, zorder=5)
    ax.scatter(x_jitter_during, during_vals, alpha=0.6, s=15, color=COLORS['teal'],
                edgecolors='black', linewidth=0.3, zorder=5)
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=6)
ax.set_yscale('symlog', linthresh=1)
ax.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([0, 1, 10, 100, 1000]))
ax.set_ylabel('Spike Rate [spikes/min; log scale]')
ax.set_ylim(bottom=0, top=20)
ax.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()
# Step 6: summary statistics
print(f"\n=== SUMMARY — MTL Stim Sz ===")
for group_name, gdf in zip(groups, group_dfs):
    print(f"\n{group_name} (n={len(gdf)}):")
    print(f"  Inter-Stim  — Median: {gdf['spike_rate_inter'].median():.2f}, Mean: {gdf['spike_rate_inter'].mean():.2f}")
    print(f"  During Stim — Median: {gdf['spike_rate_during'].median():.2f}, Mean: {gdf['spike_rate_during'].mean():.2f}")
    print(f"  Patient IDs: {sorted(gdf['subject_id'].tolist())}")
# Within-group comparisons
print(f"\n=== WITHIN-GROUP COMPARISONS (Mann-Whitney: inter vs during) ===")
for group_name, gdf in zip(groups, group_dfs):
    valid = gdf.dropna(subset=['spike_rate_inter', 'spike_rate_during'])
    if len(valid) > 1:
        stat, p = mannwhitneyu(valid['spike_rate_inter'], valid['spike_rate_during'], alternative='two-sided')
        d = cliffs_delta(valid['spike_rate_inter'].values, valid['spike_rate_during'].values)
        print(f"{group_name}: p = {p:.4f}, U = {stat:.2f}, Cliff's d = {d:.3f} (n={len(valid)})")
    else:
        print(f"{group_name}: Insufficient data (n={len(valid)})")
# Between-group comparisons
print(f"\n=== BETWEEN-GROUP COMPARISONS (Mann-Whitney: No Stim Sz vs Stim Sz) ===")
inter_no_sz  = no_stim_sz_patients['spike_rate_inter'].dropna()
inter_sz     = stim_sz_patients['spike_rate_inter'].dropna()
during_no_sz = no_stim_sz_patients['spike_rate_during'].dropna()
during_sz    = stim_sz_patients['spike_rate_during'].dropna()
if len(inter_no_sz) > 0 and len(inter_sz) > 0:
    stat, p = mannwhitneyu(inter_no_sz, inter_sz, alternative='two-sided')
    d = cliffs_delta(inter_no_sz.values, inter_sz.values)
    print(f"Inter-Stim  (No Stim Sz vs Stim Sz): p = {p:.4f}, U = {stat:.2f}, Cliff's d = {d:.3f}")
if len(during_no_sz) > 0 and len(during_sz) > 0:
    stat, p = mannwhitneyu(during_no_sz, during_sz, alternative='two-sided')
    d = cliffs_delta(during_no_sz.values, during_sz.values)
    print(f"During Stim (No Stim Sz vs Stim Sz): p = {p:.4f}, U = {stat:.2f}, Cliff's d = {d:.3f}")
print("\n" + "=" * 80 + "\n")


# Figure 4

## new packages

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from umap import UMAP
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import time

import warnings
warnings.filterwarnings("ignore")

## prep

In [ ]:
all_spikes = pd.read_csv(ospj(CONFIG.dataset_dir, 'analysis_data_morphology.csv'))
print(f'Loaded morphology data: {all_spikes.shape}')


In [ ]:
all_spikes_dist = all_spikes.copy()  # distances already merged in dataset
print(f'Morphology spikes: {all_spikes_dist.shape}')


In [ ]:
all_spikes = all_spikes_dist[all_spikes_dist['distance_mm'] <= 40].copy()

In [ ]:
all_spikes

## calculate F and save

In [ ]:
# Average morphology features per patient/stim/recording channel/condition
feature_names = ["sharpness", "width", "linelen", "slow_width",
                 "slow_amp", "rise_slope", "decay_slope", "average_amp"]
all_spikes_pt = all_spikes.groupby(
    ['subject_id', 'stim_ch', 'recording_ch', 'condition']
)[feature_names].mean().reset_index()

# Use all 8 morphology features
morph_cols = ['sharpness', 'width', 'linelen', 'slow_width',
              'slow_amp', 'rise_slope', 'decay_slope', 'average_amp']

df = all_spikes_pt.dropna(subset=morph_cols).copy().reset_index(drop=True)

# Subsample to keep distance matrix tractable
SPIKES_PER_GROUP = 200
df_sub = (df.groupby(['subject_id', 'condition'], group_keys=False)
            .apply(lambda g: g.sample(min(len(g), SPIKES_PER_GROUP), random_state=42))
            .reset_index(drop=True))

n_sub = df_sub.shape[0]
mem_gb = (n_sub ** 2 * 8) / 1e9
print(f"n after subsampling : {n_sub}")
print(f"Distance matrix size: {mem_gb:.2f} GB")
if mem_gb > 2:
    print(f"Distance matrix > 2 GB — consider lowering SPIKES_PER_GROUP. Target n_sub < 3000.")

X = StandardScaler().fit_transform(df_sub[morph_cols])

In [ ]:
# Gower-centered Gram matrix from euclidean distances
def gower_center(D):
    """O(n²) centering — avoids forming the dense n×n H matrix."""
    D2         = D ** 2
    row_mean   = D2.mean(axis=1, keepdims=True)
    col_mean   = D2.mean(axis=0, keepdims=True)
    grand_mean = D2.mean()
    return -0.5 * (D2 - row_mean - col_mean + grand_mean)

D     = pairwise_distances(X, metric='euclidean')
G     = gower_center(D)
SStot = np.trace(G)

# QR-based utilities for fast sequential SS computation
def qr_basis(Z):
    Q, _ = np.linalg.qr(Z, mode='reduced')
    return Q

def fast_trace_HG(Q, G):
    QtG = Q.T @ G
    return np.sum(QtG * Q.T)

def dummies(series):
    return pd.get_dummies(series, drop_first=True).values.astype(float)

def sequential_ss(G, SStot, design_matrices_in_order):
    n          = G.shape[0]
    intercept  = np.ones((n, 1))
    cumulative = intercept.copy()
    Q_prev     = qr_basis(cumulative)
    prev_tr    = fast_trace_HG(Q_prev, G)
    ss_list    = []
    for Z in design_matrices_in_order:
        cumulative = np.hstack([cumulative, Z])
        Q_curr     = qr_basis(cumulative)
        curr_tr    = fast_trace_HG(Q_curr, G)
        ss_list.append(curr_tr - prev_tr)
        prev_tr    = curr_tr
    ss_resid = SStot - prev_tr
    return ss_list, ss_resid, Q_curr

def f_stat(ss_term, df_term, ss_res, df_res):
    return (ss_term / df_term) / (ss_res / df_res)

# Observed F for condition term (controlling for patient + electrodes)
Z_patient   = dummies(df_sub['subject_id'])
Z_stim      = dummies(df_sub['stim_ch'])
Z_rec       = dummies(df_sub['recording_ch'])
Z_condition = dummies(df_sub['condition'])

terms  = [Z_patient, Z_stim, Z_rec, Z_condition]
labels = ['subject_id', 'stim_ch', 'recording_ch', 'condition']

ss_obs, ss_resid_obs, _ = sequential_ss(G, SStot, terms)

n        = G.shape[0]
n_terms  = [len(np.unique(df_sub[c])) - 1
            for c in ['subject_id', 'stim_ch', 'recording_ch', 'condition']]
df_resid = n - 1 - sum(n_terms)

F_obs = f_stat(ss_obs[-1], n_terms[-1], ss_resid_obs, df_resid)
print(f"\nObserved F (condition | patient + electrodes) = {F_obs:.4f}")

# Permutation test — shuffle condition within patient strata
def permute_within_strata(series, strata):
    out = series.copy()
    for s in strata.unique():
        idx = strata[strata == s].index
        out.loc[idx] = series.loc[idx].sample(frac=1, random_state=None).values
    return out

intercept   = np.ones((n, 1))
Z_nuisance  = np.hstack([intercept, Z_patient, Z_stim, Z_rec])
Q_nuisance  = qr_basis(Z_nuisance)
SS_nuisance = fast_trace_HG(Q_nuisance, G)

n_perms = 999
F_null  = []

t0 = time.time()
for i in range(n_perms):
    perm_condition = permute_within_strata(df_sub['condition'], df_sub['subject_id'])
    Z_perm  = dummies(perm_condition)
    Z_full  = np.hstack([Z_nuisance, Z_perm])
    Q_full  = qr_basis(Z_full)
    tr_full = fast_trace_HG(Q_full, G)
    ss_cond = tr_full - SS_nuisance
    ss_res  = SStot - tr_full
    F_null.append(f_stat(ss_cond, n_terms[-1], ss_res, df_resid))
    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (i + 1) * (n_perms - i - 1)
        print(f"  {i+1}/{n_perms} perms — elapsed {elapsed:.1f}s — ETA {eta:.1f}s")

F_null = np.array(F_null)
p_val  = np.mean(F_null >= F_obs)

print(f"\np-value  (condition) = {p_val:.4f}  (n_perms={n_perms})")
print(f"Pseudo-R² (condition) = {ss_obs[-1]/SStot:.4f}")
print(f"Total runtime: {time.time() - t0:.1f}s")

# Summary table
results = pd.DataFrame({
    'term': labels,
    'SS'  : ss_obs,
    'df'  : n_terms,
    'F'   : [f_stat(ss, df_, ss_resid_obs, df_resid) for ss, df_ in zip(ss_obs, n_terms)],
    'R²'  : [ss / SStot for ss in ss_obs],
}).set_index('term')
results.loc['Residual'] = [ss_resid_obs, df_resid, np.nan, ss_resid_obs / SStot]
results.loc['Total']    = [SStot, n - 1, np.nan, 1.0]
results['p_condition']  = np.nan
results.loc['condition', 'p_condition'] = p_val
results.loc['condition', 'F_obs']       = F_obs

print("\n", results.round(4))

# ── Save outputs ───────────────────────────────────────────────────────────────
results.to_csv(ospj(CONFIG.dataset_dir, 'permanova_morphology_results_40mm.csv'))
np.save(ospj(CONFIG.dataset_dir, 'permanova_F_null_40mm.npy'), F_null)
print(f"\nSaved results table → permanova_morphology_results_40mm.csv")
print(f"Saved null distribution → permanova_F_null_40mm.npy")

In [ ]:
results = pd.read_csv(ospj(CONFIG.dataset_dir, 'permanova_morphology_results_40mm.csv'), index_col=0)
F_null  = np.load(ospj(CONFIG.dataset_dir, 'permanova_F_null_40mm.npy'))

# results = pd.read_csv(ospj(CONFIG.dataset_dir, 'permanova_morphology_results.csv'), index_col=0)
# F_null  = np.load(ospj(CONFIG.dataset_dir, 'permanova_F_null.npy'))

# Reconstruct variables needed for plotting
term_rows  = ['subject_id', 'stim_ch', 'recording_ch', 'condition']

ss_obs       = results.loc[term_rows, 'SS'].tolist()
ss_resid_obs = results.loc['Residual', 'SS']
SStot        = results.loc['Total', 'SS']
n_terms      = results.loc[term_rows, 'df'].astype(int).tolist()
df_resid     = int(results.loc['Residual', 'df'])
n_sub        = int(results.loc['Total', 'df']) + 1      # Total df = n - 1
n_patients   = int(results.loc['subject_id', 'df']) + 1    # subject_id df = n_patients - 1
F_obs        = results.loc['condition', 'F_obs']
p_val        = results.loc['condition', 'p_condition']

print(f"n_sub      : {n_sub}")
print(f"n_patients : {n_patients}")
print(f"F_obs      : {F_obs:.4f}")
print(f"p_val      : {p_val}")

## plots

In [ ]:
# ── Load saved PERMANOVA results ───────────────────────────────────────────────
results = pd.read_csv(ospj(CONFIG.dataset_dir, 'permanova_morphology_results_40mm.csv'), index_col=0)
F_null  = np.load(ospj(CONFIG.dataset_dir, 'permanova_F_null_40mm.npy'))

# Reconstruct variables from saved results
term_rows    = ['subject_id', 'stim_ch', 'recording_ch', 'condition']

ss_obs       = results.loc[term_rows, 'SS'].tolist()
ss_resid_obs = results.loc['Residual', 'SS']
SStot        = results.loc['Total', 'SS']
n_terms      = results.loc[term_rows, 'df'].astype(int).tolist()
df_resid     = int(results.loc['Residual', 'df'])
n_sub        = int(results.loc['Total', 'df']) + 1
n_patients   = int(results.loc['subject_id', 'df']) + 1
F_obs        = results.loc['condition', 'F_obs']
p_val        = results.loc['condition', 'p_condition']

# Plotting variables used across all Figure 4 plot blocks
terms_fig  = ['subject_id', 'stim_ch', 'recording_ch', 'condition', 'residual']
r2         = [ss / SStot for ss in ss_obs] + [ss_resid_obs / SStot]
colors_fig = ['#4c8edd', '#2db87a', '#2db87a', '#e07b2a', '#cccccc']

def f_stat(ss_term, df_term, ss_res, df_res):
    return (ss_term / df_term) / (ss_res / df_res)

print(f"n_sub      : {n_sub}")
print(f"n_patients : {n_patients}")
print(f"F_obs      : {F_obs:.4f}")
print(f"p_val      : {p_val}")
print(f"R² condition: {r2[3]*100:.3f}%")
print(f"\nr2 by term : {[f'{v*100:.1f}%' for v in r2]}")

In [ ]:
# ── Print all text/legend content separately ──────────────────────────────────
print("=== VARIANCE PARTITION BAR — Legend ===")
for patch_label in [
    f'subject_id ({r2[0]*100:.1f}%)',
    f'stim + rec electrodes ({(r2[1]+r2[2])*100:.1f}%)',
    f'condition ({r2[3]*100:.2f}%)',
    f'residual ({r2[4]*100:.1f}%)',
]:
    print(f"  {patch_label}")

print(f"\n  Bar annotation: condition {r2[3]*100:.2f}%")
print(f"  Suptitle: n={n_sub} spikes · {n_patients} patients · 999 permutations")

print("\n=== NULL DISTRIBUTION — Legend & Annotations ===")
print(f"  null distribution (999 permutations)")
print(f"  observed F = {F_obs:.2f}")
print(f"  p = {p_val:.4f}")
print(f"  Inline text: F = {F_obs:.1f}, p < 0.001")
print(f"  Suptitle: condition: F={F_obs:.1f}, R²={ss_obs[-1]/SStot:.4f}, p<0.001")

# ── Plot 1: Variance partition bar — x-axis only ──────────────────────────────
colors_fig_split = ['#4c8edd', '#e6b800', '#16a085', '#e07b2a', '#cccccc']
fig1, ax = plt.subplots(figsize=(70*MM_TO_IN, 20*MM_TO_IN))
left = 0
bar_height = 0.5
COND_MIN = 0.010
for term, val, col in zip(terms_fig, r2, colors_fig_split):
    draw_width = max(val, COND_MIN) if col == '#e07b2a' else val
    ax.barh(0, draw_width, left=left, height=bar_height,
            color=col, edgecolor='white', linewidth=1.5)
    left += draw_width  # advance by drawn width so residual starts after the sliver
ax.set_xlim(0, 1)
ax.set_ylim(-bar_height/2, bar_height/2)
ax.set_yticks([])
ax.set_xlabel('Proportion of total variance (R²)', fontsize=8)
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

# ── Plot 2: Broken x-axis — null distribution vs observed F ───────────────────
fig2, (ax_null, ax_obs) = plt.subplots(
    1, 2, figsize=(70*MM_TO_IN, 50*MM_TO_IN), sharey=True,
    gridspec_kw={'width_ratios': [5, 1], 'wspace': 0.04}
)
ax_null.hist(F_null, bins=50, color='red', edgecolor='white',
             linewidth=0.5, density=True)
ax_null.set_xlim(0, np.percentile(F_null, 99) * 1.15)
ax_null.set_xlabel('Pseudo-F (condition)', fontsize=7)
ax_null.set_ylabel('Density', fontsize=7)
ax_null.tick_params(labelsize=6)
ax_null.spines[['top', 'right']].set_visible(False)
ax_obs.axvline(F_obs, color='#e07b2a', linewidth=1.5, linestyle='--', zorder=5)
ax_obs.set_xlim(F_obs - 2, F_obs + 2)
ax_obs.set_xticks([F_obs])
ax_obs.set_xticklabels([f'{F_obs:.1f}'], fontsize=6, color='#e07b2a', rotation=45)
ax_obs.tick_params(labelsize=6)
ax_obs.spines[['top', 'right', 'left']].set_visible(False)
ax_obs.set_yticks([])
# Bottom break marks only
d = 0.03
kw = dict(color='k', clip_on=False, linewidth=0.8, transform=ax_null.transAxes)
ax_null.plot([1-d, 1+d], [-d, +d], **kw)
kw.update(transform=ax_obs.transAxes)
ax_obs.plot([-d, +d], [-d, +d], **kw)
plt.tight_layout()
plt.show()


In [ ]:
# Residualize morphology features against patient + electrode nuisance terms
nuisance_dummies = pd.get_dummies(
    df_sub[['subject_id', 'stim_ch', 'recording_ch']], drop_first=True
).values.astype(float)

X_scaled = StandardScaler().fit_transform(df_sub[morph_cols])
reg      = LinearRegression(fit_intercept=True).fit(nuisance_dummies, X_scaled)
X_resid  = X_scaled - reg.predict(nuisance_dummies)

pca   = PCA().fit(X_resid)
X_pca = pca.transform(X_resid)
print("Cumulative variance explained by PC1-5:")
print(np.cumsum(pca.explained_variance_ratio_[:5]).round(3))

colors_cond = {'during': 'red', 'inter': 'blue'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Spike morphology after removing patient + electrode effects',
             fontsize=11, y=1.01)

for ax, (pc_x, pc_y) in zip(axes, [(0, 1), (0, 2)]):
    for cond, grp in df_sub.groupby('condition'):
        idx = grp.index
        ax.scatter(X_pca[idx, pc_x], X_pca[idx, pc_y],
                   c=colors_cond[cond], label=cond, alpha=0.3, s=4, linewidths=0)

    for cond, grp in df_sub.groupby('condition'):
        idx  = grp.index
        mu   = X_pca[idx][:, [pc_x, pc_y]].mean(axis=0)
        cov  = np.cov(X_pca[idx][:, [pc_x, pc_y]].T)
        vals, vecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(*vecs[:, -1][::-1]))
        for n_std, alpha in [(1, 0.3), (2, 0.15)]:
            ell = Ellipse(xy=mu, width=2*n_std*np.sqrt(vals[0]),
                          height=2*n_std*np.sqrt(vals[1]),
                          angle=angle, color=colors_cond[cond],
                          alpha=alpha, zorder=3)
            ax.add_patch(ell)
        ax.scatter(*mu, c=colors_cond[cond], s=60, zorder=4,
                   edgecolors='white', linewidths=0.8)

    ax.set_xlabel(f'PC{pc_x+1} ({pca.explained_variance_ratio_[pc_x]*100:.1f}%)')
    ax.set_ylabel(f'PC{pc_y+1} ({pca.explained_variance_ratio_[pc_y]*100:.1f}%)')
    ax.legend(title='condition', markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# Residualize morphology features against patient + electrode nuisance terms
nuisance_dummies = pd.get_dummies(
    df_sub[['subject_id', 'stim_ch', 'recording_ch']], drop_first=True
).values.astype(float)

X_scaled = StandardScaler().fit_transform(df_sub[morph_cols])
reg      = LinearRegression(fit_intercept=True).fit(nuisance_dummies, X_scaled)
X_resid  = X_scaled - reg.predict(nuisance_dummies)

# Fit UMAP on residualized morphology
reducer = UMAP(n_neighbors=30, min_dist=0.3, random_state=42, n_jobs=1)
X_umap  = reducer.fit_transform(X_resid)

colors_cond = {'during': 'red', 'inter': 'blue'}

fig, ax = plt.subplots(figsize=(50*MM_TO_IN, 50*MM_TO_IN))

for cond, grp in df_sub.groupby('condition'):
    idx = grp.index
    ax.scatter(X_umap[idx, 0], X_umap[idx, 1],
               c=colors_cond[cond], label=cond, alpha=0.5, s=4, linewidths=0)

ax.set_xlabel('UMAP 1', fontsize=7)
ax.set_ylabel('UMAP 2', fontsize=7)
ax.tick_params(labelsize=6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-feature mean difference (during − inter) with 95% bootstrap CI ────────
n_boot = 2000
rng    = np.random.default_rng(42)

during_idx = df_sub.index[df_sub['condition'] == 'during']
inter_idx  = df_sub.index[df_sub['condition'] == 'inter']

X_resid_df             = pd.DataFrame(X_resid, index=df_sub.index, columns=morph_cols)
X_resid_df['condition'] = df_sub['condition'].values

feat_labels = {
    'sharpness'  : 'Sharpness',
    'width'      : 'Width',
    'linelen'    : 'Line length',
    'slow_width' : 'Slow width',
    'slow_amp'   : 'Slow amplitude',
    'rise_slope' : 'Rise slope',
    'decay_slope': 'Decay slope',
    'average_amp': 'Amplitude',
}

diffs, ci_lo, ci_hi = [], [], []

for col in morph_cols:
    d_vals   = X_resid_df.loc[during_idx, col].values
    i_vals   = X_resid_df.loc[inter_idx,  col].values
    obs_diff = d_vals.mean() - i_vals.mean()
    boot_diffs = np.array([
        rng.choice(d_vals, len(d_vals), replace=True).mean() -
        rng.choice(i_vals, len(i_vals), replace=True).mean()
        for _ in range(n_boot)
    ])
    diffs.append(obs_diff)
    ci_lo.append(np.percentile(boot_diffs, 2.5))
    ci_hi.append(np.percentile(boot_diffs, 97.5))

diffs  = np.array(diffs)
ci_lo  = np.array(ci_lo)
ci_hi  = np.array(ci_hi)

order       = np.argsort(diffs)
feat_sorted = [feat_labels[morph_cols[i]] for i in order]
d_sorted    = diffs[order]
lo_sorted   = ci_lo[order]
hi_sorted   = ci_hi[order]
sig_sorted  = [not (lo <= 0 <= hi) for lo, hi in zip(lo_sorted, hi_sorted)]

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(60*MM_TO_IN, 33*MM_TO_IN))

y_pos      = np.arange(len(morph_cols))
bar_colors = ['red' if d > 0 else 'blue' for d in d_sorted]
bar_alphas = [0.85 if s else 0.35 for s in sig_sorted]

for y, d, lo, hi, col, alpha, sig in zip(
        y_pos, d_sorted, lo_sorted, hi_sorted, bar_colors, bar_alphas, sig_sorted):
    ax.barh(y, d, color=col, alpha=alpha, height=0.55)
    ax.errorbar(d, y, xerr=[[d - lo], [hi - d]],
                fmt='none', color='#333333', linewidth=0.7,
                capsize=1.5, capthick=0.7)
    if sig:
        x_pos = hi + 0.005 if d > 0 else lo - 0.005
        ha    = 'left' if d > 0 else 'right'
        ax.text(x_pos, y, '*', fontsize=7, color='#333333',
                ha=ha, va='center')

ax.axvline(0, color='#333333', linewidth=0.6, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_sorted, fontsize=7)
# ax.set_xlabel('Mean difference (during − inter)\nresidualised z-score', fontsize=6)
ax.tick_params(labelsize=5, length=2)
ax.spines[['top', 'right']].set_visible(False)
# plt.tight_layout()
ax.set_xlim(-0.15,0.26)
plt.show()

print(f"{'Feature':<16} {'Diff':>8} {'CI_lo':>8} {'CI_hi':>8}")
for f, d, lo, hi, sig in zip(feat_sorted, d_sorted, lo_sorted, hi_sorted, sig_sorted):
    print(f"{f:<16} {d:>8.4f} {lo:>8.4f} {hi:>8.4f}  {'*' if sig else ''}")

In [ ]:
# ── Per-feature mean difference (during − inter) with 95% bootstrap CI ────────
n_boot = 2000
rng    = np.random.default_rng(42)

during_idx = df_sub.index[df_sub['condition'] == 'during']
inter_idx  = df_sub.index[df_sub['condition'] == 'inter']

X_resid_df              = pd.DataFrame(X_resid, index=df_sub.index, columns=morph_cols)
X_resid_df['condition'] = df_sub['condition'].values

feat_labels = {
    'sharpness'  : 'Sharpness',
    'width'      : 'Width',
    'linelen'    : 'Line length',
    'slow_width' : 'Slow width',
    'slow_amp'   : 'Slow amplitude',
    'rise_slope' : 'Rise slope',
    'decay_slope': 'Decay slope',
    'average_amp': 'Amplitude',
}

diffs, ci_lo, ci_hi = [], [], []

for col in morph_cols:
    d_vals   = X_resid_df.loc[during_idx, col].values
    i_vals   = X_resid_df.loc[inter_idx,  col].values
    obs_diff = d_vals.mean() - i_vals.mean()
    boot_diffs = np.array([
        rng.choice(d_vals, len(d_vals), replace=True).mean() -
        rng.choice(i_vals, len(i_vals), replace=True).mean()
        for _ in range(n_boot)
    ])
    diffs.append(obs_diff)
    ci_lo.append(np.percentile(boot_diffs, 2.5))
    ci_hi.append(np.percentile(boot_diffs, 97.5))

diffs  = np.array(diffs)
ci_lo  = np.array(ci_lo)
ci_hi  = np.array(ci_hi)

order       = np.argsort(diffs)
feat_sorted = [feat_labels[morph_cols[i]] for i in order]
d_sorted    = diffs[order]
lo_sorted   = ci_lo[order]
hi_sorted   = ci_hi[order]
sig_sorted  = [not (lo <= 0 <= hi) for lo, hi in zip(lo_sorted, hi_sorted)]

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(50*MM_TO_IN, 50*MM_TO_IN))

x_pos      = np.arange(len(morph_cols))
bar_colors = ['red' if d > 0 else 'blue' for d in d_sorted]

for x, d, lo, hi, col, sig in zip(
        x_pos, d_sorted, lo_sorted, hi_sorted, bar_colors, sig_sorted):
    ax.bar(x, d, color=col, alpha=0.85 if sig else 0.35, width=0.55)
    ax.errorbar(x, d, yerr=[[d - lo], [hi - d]],
                fmt='none', color='#333333', linewidth=0.7,
                capsize=1.5, capthick=0.7)
    if sig:
        y_pos = hi + 0.005 if d > 0 else lo - 0.005
        va    = 'bottom' if d > 0 else 'top'
        ax.text(x, y_pos, '*', fontsize=7, color='#333333',
                ha='center', va=va)

ax.axhline(0, color='#333333', linewidth=0.6, linestyle='--')
ax.set_xticks(x_pos)
ax.set_xticklabels(feat_sorted, fontsize=7, rotation=45, ha='right')
ax.tick_params(labelsize=6, length=2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f"{'Feature':<16} {'Diff':>8} {'CI_lo':>8} {'CI_hi':>8}")
for f, d, lo, hi, sig in zip(feat_sorted, d_sorted, lo_sorted, hi_sorted, sig_sorted):
    print(f"{f:<16} {d:>8.4f} {lo:>8.4f} {hi:>8.4f}  {'*' if sig else ''}")

## inter vs. during plot of unique spike morphology

In [ ]:
# ── Cell 3 (updated): separate figures, 50×50 mm each ────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

MORPH_FEATURES = ['slow_amp', 'slow_width', 'rise_slope', 'decay_slope',
                  'average_amp', 'linelen', 'sharpness', 'width']

combined = pd.concat([
    during_sample.assign(group='during'),
    inter_sample.assign(group='inter')
]).reset_index(drop=True)

X_morph        = combined[MORPH_FEATURES].values.astype(float)
scaler         = StandardScaler()
X_morph_scaled = scaler.fit_transform(X_morph)
pca_morph      = PCA(n_components=2)
Z_morph        = pca_morph.fit_transform(X_morph_scaled)
group_labels   = combined['group'].values
colors         = {'during': 'firebrick', 'inter': 'steelblue'}
legend_handles = {'During-': 'firebrick', 'Inter-': 'steelblue'}

# ── (A) Morphological PCA ──────────────────────────────────────────────────────
fig_a, ax = plt.subplots(figsize=(40 * MM_TO_IN, 40 * MM_TO_IN))

for grp, color in colors.items():
    mask = group_labels == grp
    ax.scatter(Z_morph[mask, 0], Z_morph[mask, 1],
               c=color, alpha=1, s=8, label=grp, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca_morph.explained_variance_ratio_[0]*100:.1f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pca_morph.explained_variance_ratio_[1]*100:.1f}%)', fontsize=7)
# ax.set_title('PCA – Morphological features', fontsize=5)
ax.legend(handles=[mpatches.Patch(color=c, label=g) for g, c in legend_handles.items()],
          fontsize=6, markerscale=0.6)
ax.tick_params(labelsize=4)
ax.spines[['top', 'right']].set_visible(False)
# ax.axhline(0, color='grey', linewidth=0.4, linestyle='--')
# ax.axvline(0, color='grey', linewidth=0.4, linestyle='--')

plt.tight_layout()
plt.show()

# ── (B) Sharpness vs. Width scatter ───────────────────────────────────────────
fig_b, ax = plt.subplots(figsize=(50 * MM_TO_IN, 50 * MM_TO_IN))

for grp, color in colors.items():
    mask = group_labels == grp
    ax.scatter(combined.loc[mask, 'width'],
               combined.loc[mask, 'sharpness'],
               c=color, alpha=0.8, s=8, edgecolors='none', label=grp)

ax.set_xlabel('Width (ms)', fontsize=5)
ax.set_ylabel('Sharpness (a.u.)', fontsize=5)
ax.set_title('Spike Sharpness vs. Width', fontsize=5)
ax.legend(handles=[mpatches.Patch(color=c, label=g) for g, c in colors.items()],
          fontsize=4, markerscale=0.6)
ax.tick_params(labelsize=4)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary ────────────────────────────────────────────────────────────────────
print("=== Morphological PCA ===")
for i, (ev, feat_load) in enumerate(zip(pca_morph.explained_variance_ratio_,
                                        pca_morph.components_)):
    top     = sorted(zip(feat_load, MORPH_FEATURES), key=lambda x: abs(x[0]), reverse=True)[:3]
    top_str = ', '.join(f'{f} ({v:+.2f})' for v, f in top)
    print(f"  PC{i+1}: {ev*100:.1f}%  |  top loadings: {top_str}")

# Figure 5 

look at @inter_ML.py and @inter_ML_fixed.py
- inter_ML.py does independent optmization for each model
- inter_ML_fixed.py does a fixed model approach where each gets the same apriori hyperparameters